# Thesis figures, one per cell

Each figure below is drawn by its own cell, reading its own data straight out of the committed
result files under `results/accord/`. No cell depends on any variable another figure's cell
created; each one loads what it needs, prints the values it is about to plot, draws, displays the
figure inline, and saves it as a PDF.

Run cells 1 to 3 first (setup, palette, small shared helpers). After that, every figure cell below
runs on its own, in any order.

**No credentialed data is shipped here.** MIMIC-III is a PhysioNet-credentialed dataset; every
result file this notebook reads is an aggregate (counts, rates, confusion-matrix cells, bootstrap
intervals) with no clinical text and no patient or admission identifiers.

## 1. Setup

In [1]:
%matplotlib inline
import os, sys, shutil, json, pathlib, math, statistics
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import norm
from scipy.optimize import brentq
import pandas as pd

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BUNDLE = "/content/drive/MyDrive/This is paper/02_figures/colab_bundle"
    REPO = "/content/repo"
    os.makedirs(REPO, exist_ok=True)
    dst_results = os.path.join(REPO, "results")
    if os.path.exists(dst_results):
        shutil.rmtree(dst_results)
    shutil.copytree(os.path.join(DRIVE_BUNDLE, "results"), dst_results)
    print(f"copied bundle results: {DRIVE_BUNDLE}/results -> {dst_results}")
else:
    # Local run: this machine already has the repository checked out, so there is
    # nothing to mount or copy. REPO points straight at it.
    REPO = "/Users/rickygole/provenance-encoder"
    print(f"not running in Colab; reading directly from the local repository at {REPO}")

RESULTS = os.path.join(REPO, "results")
RESULTS_ACCORD = os.path.join(RESULTS, "accord")
FIGDIR = os.path.join(REPO, "accord", "thesis", "figures")
os.makedirs(FIGDIR, exist_ok=True)

# Defensive only: no cell in this notebook resamples or fits anything. Every
# bootstrap interval plotted below was already computed upstream (the seed is
# recorded inside the source JSON, typically 42) and is simply read and plotted.
np.random.seed(42)

n_json = sum(1 for _ in pathlib.Path(RESULTS_ACCORD).rglob("*.json"))
print("REPO           =", REPO)
print("RESULTS_ACCORD =", RESULTS_ACCORD)
print("FIGDIR         =", FIGDIR)
print("json files under results/accord:", n_json)

not running in Colab; reading directly from the local repository at /Users/rickygole/provenance-encoder
REPO           = /Users/rickygole/provenance-encoder
RESULTS_ACCORD = /Users/rickygole/provenance-encoder/results/accord
FIGDIR         = /Users/rickygole/provenance-encoder/accord/thesis/figures
json files under results/accord: 99


## 2. Shared palette

Four colours plus a red accent, used by name in every figure cell below (`DARK`, `MID`, `FILL`,
`DEAD`, `RED`). They differ in **lightness**, not just hue, so the figures still separate correctly
in a black-and-white print; the readout below prints that separation. This is the same palette
already established for this bundle in `Thesis_Figures_Restyle.ipynb`.

In [ ]:
# >>> generated from style.py
# style.py sections 1-6: the ramps, the structure tones, the entity map,
# the type scale, the line weights and the rcParams. Copied verbatim, so
# tools/check_parity.py can assert set-equality with the module itself.
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patheffects as mpe
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Rectangle, ArrowStyle

# ---------------------------------------------------------------------------
# 1. THE RAMPS
# ---------------------------------------------------------------------------
class Ramp:
    """One hue, four stops plus BARE. Immutable by convention."""

    __slots__ = ("name", "DEEP", "CORE", "TINT", "WASH", "BARE")

    def __init__(self, name, deep, core, tint, wash, bare):
        self.name = name
        self.DEEP, self.CORE, self.TINT, self.WASH, self.BARE = (
            deep, core, tint, wash, bare)

    def __repr__(self):
        return f"<Ramp {self.name}>"

    def asdict(self):
        return {"DEEP": self.DEEP, "CORE": self.CORE, "TINT": self.TINT,
                "WASH": self.WASH, "BARE": self.BARE}


#                name       DEEP       CORE       TINT       WASH       BARE
COBALT = Ramp("COBALT",  "#0A4DB1", "#0B53C0", "#BDD0EE", "#DEE8F7", "#98B6E4")
AZURE = Ramp("AZURE",    "#20577D", "#3FA9F5", "#A1D5FA", "#D1EBFD", "#78C2F8")
EMERALD = Ramp("EMERALD", "#005F44", "#00A97A", "#98DCC9", "#CEEFE5", "#75D0B7")
AMBER = Ramp("AMBER",    "#714B00", "#F5A300", "#F9C765", "#FCE4B6", "#F6AC1A")
PLUM = Ramp("PLUM",      "#971D6B", "#C3268A", "#EEC2DE", "#F7E1EF", "#E296C6")
SCARLET = Ramp("SCARLET", "#A41704", "#D81E05", "#F5C2BC", "#FAE2DF", "#EE9A8F")

RAMPS = {r.name: r for r in (COBALT, AZURE, EMERALD, AMBER, PLUM, SCARLET)}

# Flat names, because a drawing script reads better as SCARLET_DEEP than as
# SCARLET.DEEP inside an f-string-heavy layout block.
COBALT_DEEP, COBALT_CORE, COBALT_TINT, COBALT_WASH, COBALT_BARE = (
    COBALT.DEEP, COBALT.CORE, COBALT.TINT, COBALT.WASH, COBALT.BARE)
AZURE_DEEP, AZURE_CORE, AZURE_TINT, AZURE_WASH, AZURE_BARE = (
    AZURE.DEEP, AZURE.CORE, AZURE.TINT, AZURE.WASH, AZURE.BARE)
EMERALD_DEEP, EMERALD_CORE, EMERALD_TINT, EMERALD_WASH, EMERALD_BARE = (
    EMERALD.DEEP, EMERALD.CORE, EMERALD.TINT, EMERALD.WASH, EMERALD.BARE)
AMBER_DEEP, AMBER_CORE, AMBER_TINT, AMBER_WASH, AMBER_BARE = (
    AMBER.DEEP, AMBER.CORE, AMBER.TINT, AMBER.WASH, AMBER.BARE)
PLUM_DEEP, PLUM_CORE, PLUM_TINT, PLUM_WASH, PLUM_BARE = (
    PLUM.DEEP, PLUM.CORE, PLUM.TINT, PLUM.WASH, PLUM.BARE)
SCARLET_DEEP, SCARLET_CORE, SCARLET_TINT, SCARLET_WASH, SCARLET_BARE = (
    SCARLET.DEEP, SCARLET.CORE, SCARLET.TINT, SCARLET.WASH, SCARLET.BARE)

# ---------------------------------------------------------------------------
# 2. STRUCTURE TONES. No identity, ever.
# ---------------------------------------------------------------------------
PAPER = "#FFFFFF"      # 1.0000
INK = "#16181D"        # 0.0091  17.76:1   text, spines, ticks
GRAPHITE = "#2E3440"   # 0.0341  12.49:1   connectors; a MEASURED non-entity
MUTE = "#5A6072"       # 0.1175   6.27:1   secondary annotation text
SLATE = "#7C8AA5"      # 0.2518   3.48:1   reference lines, seed dots, stubs
MIST = "#DCE3EE"       # 0.7633            neutral structural ground
HAIR = "#EAEDF3"       # 0.8453            the one optional value grid
PLATE_EDGE = "#C4CCDA" # schematic enclosure stroke only

STRUCTURE = {"PAPER": PAPER, "INK": INK, "GRAPHITE": GRAPHITE, "MUTE": MUTE,
             "SLATE": SLATE, "MIST": MIST, "HAIR": HAIR,
             "PLATE_EDGE": PLATE_EDGE}

# The single machine-readable source of every hex in the document.
PALETTE = {name: r.asdict() for name, r in RAMPS.items()}
PALETTE["STRUCTURE"] = dict(STRUCTURE)

# ---------------------------------------------------------------------------
# 3. THE ENTITY MAP. An entity that appears on more than one figure is the same
# hue on every one of them.
# ---------------------------------------------------------------------------
ENTITY_HUE = {
    "gpt4o_zeroshot":       COBALT,
    "gpt4o_fewshot":        AZURE,    # same model, other prompt -> same family
    "accord":               AMBER,
    "llama3_8b":            PLUM,
    "weak_labeller":        PLUM,     # the production weak labeller IS Llama 3 8B
    "pharmacist_truth":     EMERALD,
    "pretrained_reference": EMERALD,
    "qwen3_8b":             COBALT,   # no GPT-4o series on that canvas
    "granite33_8b":         AMBER,    # no ACCORD series on that canvas
    "control_model":        EMERALD,
}
# Alpha parameters, the MDE curve, coded-corpus counts: measured, but not an
# entity, so they take the structural measurement tone rather than a hue.
NON_ENTITY_MEASUREMENT = GRAPHITE

# ---------------------------------------------------------------------------
# 4. TYPE SCALE. Every figure is drawn at its printed size (scale 1.0), so a
# drawn point is a page point.
# ---------------------------------------------------------------------------
T_HEADLINE = 13.0      # INK, regular. A finding, never a title.
T_CLAIM = 22.0         # claim DEEP, bold. Only where a directive says so.
T_DECK = 9.5           # MUTE, regular
T_AXIS = 9.5           # INK
T_TICK = 9.0           # INK
T_SERIES = 9.0         # that series' DEEP
T_NOTE = 8.5           # MUTE, italic
T_PANEL = 10.0         # INK, italic
T_CHIP = 9.0           # PAPER, bold
T_FLOOR = 8.5

# ---------------------------------------------------------------------------
# 5. LINE WEIGHTS. The whole set, no exceptions.
# ---------------------------------------------------------------------------
LW_SIGNATURE = 3.0     # bar height in points
LW_PLATE = 1.0
LW_SPINE = 0.9
LW_GRID = 0.6
LW_DATA = 1.8
LW_REF = 1.0
LW_INTERVAL = 1.5
LW_NODE = 1.5          # F1 schematic node stroke
LW_CHIP = 1.4          # F2 solid chip rim
LW_CONNECTOR = 1.4
LW_BAR = 1.0           # bar outline, NEW in v3

MS_DOT = 6.6
MS_MARK = 6.2          # marker, non-dot
MS_BARE = 6.6          # open, 1.2pt edge in X_BARE
MS_SEED = 3.2          # SLATE, solid

REF_DASH = (0, (4, 2.4))
ERRBAR = dict(elinewidth=LW_INTERVAL, capthick=LW_INTERVAL, capsize=3.2)
DOT = dict(markersize=MS_DOT)

# ---------------------------------------------------------------------------
# 6. rcParams
# ---------------------------------------------------------------------------
RCPARAMS = {
    "font.family": "serif",
    # STIXGeneral is Times-metric-compatible and ships inside matplotlib's own
    # mpl-data/fonts/ttf, so it renders identically here and in Colab, which
    # was the entire reason DejaVu was pinned. It is also narrower than DejaVu,
    # so every spacing decision already checked has more room, not less.
    "font.serif": ["STIXGeneral"],
    "mathtext.fontset": "stix",
    "font.size": T_TICK,
    "axes.labelsize": T_AXIS,
    "axes.titlesize": T_AXIS,
    "xtick.labelsize": T_TICK,
    "ytick.labelsize": T_TICK,
    "legend.fontsize": T_TICK,
    "pdf.fonttype": 42,
    # A negative tick label must be a hyphen-minus, not U+2212.
    "axes.unicode_minus": False,
    "text.color": INK,
    "axes.labelcolor": INK,
    "axes.edgecolor": INK,
    "xtick.color": INK,
    "ytick.color": INK,
    # Boxed. All four spines.
    "axes.spines.top": True,
    "axes.spines.right": True,
    "axes.spines.left": True,
    "axes.spines.bottom": True,
    "axes.linewidth": LW_SPINE,
    "xtick.major.width": LW_SPINE,
    "ytick.major.width": LW_SPINE,
    "xtick.major.size": 3.2,
    "ytick.major.size": 3.2,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "axes.grid": False,
    "grid.linewidth": LW_GRID,
    "grid.color": HAIR,
    "lines.linewidth": LW_DATA,
    "legend.frameon": False,
    "figure.dpi": 150,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
}


def setup():
    plt.rcParams.update(RCPARAMS)

setup()
# <<< generated from style.py


## 3. Shared helpers

Small and genuinely reused across every figure cell below: the print-legible style (serif, small
fonts, embedded PDF fonts, no top/right spines), a dashed "chance" line style used by more than one
figure, and a `save_fig` function that writes the PDF and reports its path. Anything used by only
one figure is defined inside that figure's own cell instead, not here.

In [ ]:
# >>> generated from style.py
# style.py sections 7-10: measure, the frame system, marks and grounds,
# and the one interval this set is allowed to compute.
# ---------------------------------------------------------------------------
# 7. MEASURE
# ---------------------------------------------------------------------------
# \textwidth is 433.62pt = 6.014in. Two width classes only, 0.90 and 0.80.
TEXTWIDTH_IN = 6.014
FIGW = 5.36            # the 0.90 class, drawn
PRINT_W = 5.40


def figw(frac):
    """(drawn_width, printed_width) for a given \\linewidth fraction.

    savefig pad is 0.02in per side and canvas() pins the bbox, so
    saved width = drawn + 0.04 = printed exactly.
    """
    printed = frac * TEXTWIDTH_IN
    return printed - 0.04, printed


W90, _ = figw(0.90)    # 5.3726 -> the roster's 5.36 rounds to this
W80, _ = figw(0.80)    # 4.7712


def canvas(fig):
    """Pin the saved bounding box to the whole figure.

    savefig crops to the artists, so the width of the file would otherwise
    depend on how far the longest annotation happened to reach and no two
    figures would come out the same measure.
    """
    fig.patches.append(Rectangle((0, 0), 1, 1, transform=fig.transFigure,
                                 fill=False, linewidth=0, edgecolor="none"))


# ---------------------------------------------------------------------------
# 8. THE FRAME SYSTEM. Two enclosures, exactly one per figure.
# ---------------------------------------------------------------------------
def frame(ax, grid=False):
    """Full boxed axes. Replaces v2's lean().

    All four spines at 0.9pt INK, ticks outward. set_bounds() clipping is
    removed everywhere: a closed box cannot be clipped and the two ideas fight.

    grid draws the one optional value grid: True for both axes, or "x" / "y"
    for the one axis the plot is read along. The rule the builder does NOT get
    to decide: a plot read for VALUES gets the grid; a plot read for position
    or shape does not. It is specified per figure.
    """
    for s in ("top", "right", "left", "bottom"):
        ax.spines[s].set_visible(True)
        ax.spines[s].set_linewidth(LW_SPINE)
        ax.spines[s].set_color(INK)
    if grid:
        ax.set_axisbelow(True)
        ax.grid(True, axis="both" if grid is True else grid, color=HAIR,
                linewidth=LW_GRID, zorder=0)
    return ax


def plate(fig):
    """The schematic enclosure: a rounded rectangle covering the full canvas,
    inset 1.5pt from every edge, corner radius 4pt, PAPER fill, PLATE_EDGE
    stroke at 1.0pt. No inner axes box. A plot never gets one."""
    w_in, h_in = fig.get_size_inches()
    ix, iy = 1.5 / (72 * w_in), 1.5 / (72 * h_in)
    p = FancyBboxPatch(
        (ix, iy), 1 - 2 * ix, 1 - 2 * iy,
        boxstyle="round,pad=0,rounding_size=0.010",
        transform=fig.transFigure, facecolor=PAPER, edgecolor=PLATE_EDGE,
        linewidth=LW_PLATE, mutation_aspect=w_in / h_in, zorder=-10,
        clip_on=False)
    fig.patches.append(p)
    return p


def signature(fig, hue_deep, headline=None, deck=None, headline_x=None):
    """The rule that makes twelve figures one set.

    A 3.0pt-tall solid bar flush to the top inside edge of the canvas, running
    its full width, in this figure's dominant DEEP hue. The headline sits 7pt
    below it: one line stating what the figure FOUND, never what it plots, at
    13pt against a 12pt body, so the figure stops reading as an inclusion.
    """
    h_in = fig.get_size_inches()[1]
    bar_h = LW_SIGNATURE / (72 * h_in)
    fig.patches.append(Rectangle((0, 1 - bar_h), 1, bar_h,
                                 transform=fig.transFigure, facecolor=hue_deep,
                                 edgecolor="none", zorder=20, clip_on=False))
    y = 1 - (LW_SIGNATURE + 7.0) / (72 * h_in)
    x = 0.012 if headline_x is None else headline_x
    if headline:
        fig.text(x, y, headline, ha="left", va="top", fontsize=T_HEADLINE,
                 color=INK, zorder=21)
        y -= (T_HEADLINE * 1.06) / (72 * h_in)
    if deck:
        fig.text(x, y, deck, ha="left", va="top", fontsize=T_DECK, color=MUTE,
                 zorder=21)
    return y


def panel_label(fig, ax, text):
    """Italic 10pt INK in FIGURE coordinates: x at the axes box's left edge,
    y 5pt above the axes box top edge. Always outside the frame, never inside,
    never in data coordinates. No figure computes this itself."""
    h_in = fig.get_size_inches()[1]
    p = ax.get_position()
    return fig.text(p.x0, p.y1 + 5.0 / (72 * h_in), text, ha="left",
                    va="bottom", fontsize=T_PANEL, style="italic", color=INK,
                    zorder=21)


# ---------------------------------------------------------------------------
# 9. MARKS AND GROUNDS
# ---------------------------------------------------------------------------
# matplotlib's head_width is a HALF-width, and the 1.4pt stroke adds about
# 1.9pt to the rendered width, so the spec's literal (0.62, 0.30) at
# mutation_scale 12 renders 7.2pt long by 9.4pt wide and misses the 6.0-8.5pt
# width gate. These two numbers are the ones that hit the spec's stated target
# of about 7.4pt long by 7.2pt wide: measured 7.2 x 7.6 at scale 12.
_ARROWSTYLE = ArrowStyle("-|>", head_length=0.62, head_width=0.24)


def arrow(ax, xy_from, xy_to, color=GRAPHITE, lw=LW_CONNECTOR, zorder=3,
          mutation_scale=12, **kw):
    """One arrowhead for the whole set. matplotlib's default micro-heads are
    replaced everywhere: this targets a head about 7.4pt long and 7.2pt wide,
    solid-filled in the stroke colour. No script builds one by hand."""
    a = FancyArrowPatch(xy_from, xy_to, arrowstyle=_ARROWSTYLE,
                        mutation_scale=mutation_scale, lw=lw, shrinkA=0,
                        shrinkB=0, joinstyle="miter", capstyle="butt",
                        color=color, zorder=zorder, **kw)
    ax.add_patch(a)
    return a


def chip(ax, x, y, w, h, hue, text, fontsize=T_CHIP, radius=3.0, zorder=6,
         ha="center", stroke=None):
    """F2 SOLID CHIP: X_DEEP fill, X_CORE rim at 1.4pt, PAPER text, bold.

    hue may be a Ramp or a (fill, rim) pair, so the SLATE/MIST no-verdict state
    can use the same helper without inventing a seventh ramp.
    """
    if isinstance(hue, Ramp):
        fill, rim = hue.DEEP, hue.CORE
    else:
        fill, rim = hue
    if stroke is not None:
        rim = stroke
    ax.add_patch(FancyBboxPatch(
        (x, y), w, h,
        boxstyle=f"round,pad=0,rounding_size={radius}", mutation_aspect=1.0,
        linewidth=LW_CHIP, edgecolor=rim, facecolor=fill, zorder=zorder,
        clip_on=False))
    tx = x + w / 2 if ha == "center" else x + 6.0
    t = ax.text(tx, y + h / 2, text, ha=ha, va="center", fontsize=fontsize,
                color=PAPER, fontweight="bold", zorder=zorder + 1,
                clip_on=False)
    return t


def node(ax, x, y, w, h, hue_tint, hue_deep, radius=3.0, zorder=4, lw=LW_NODE):
    """F1 TINT PANEL: X_TINT fill, X_DEEP stroke 1.5pt. Text is drawn by the
    caller in INK, because a node's text is sometimes two lines at two sizes."""
    p = FancyBboxPatch(
        (x, y), w, h,
        boxstyle=f"round,pad=0,rounding_size={radius}", mutation_aspect=1.0,
        linewidth=lw, edgecolor=hue_deep, facecolor=hue_tint, zorder=zorder,
        clip_on=False)
    ax.add_patch(p)
    return p


def band(ax, x0, x1, tint, zorder=0, axis="x"):
    """F3 WASH/TINT GROUND spanning the full extent of the other axis. No
    stroke, no alpha: the tint hex is precomputed."""
    if axis == "x":
        return ax.axvspan(x0, x1, facecolor=tint, edgecolor="none",
                          linewidth=0, zorder=zorder)
    return ax.axhspan(x0, x1, facecolor=tint, edgecolor="none", linewidth=0,
                      zorder=zorder)


def case(artist, linewidth=2.8):
    """Casing: any text or line sitting on a filled field takes a white stroke
    under it, so it reads against the fill and against the paper both. Applies
    over TINT, WASH and heatmap fields only. Text on a SOLID chip is white and
    is never cased."""
    fx = [mpe.withStroke(linewidth=linewidth, foreground=PAPER)]
    try:
        artist.set_path_effects(fx)
    except AttributeError:
        for a in artist:
            a.set_path_effects(fx)
    return artist


def cased(fontsize=T_SERIES, linewidth=2.8):
    """Kwargs form, for the common `ax.text(..., **style.cased())` call."""
    return dict(fontsize=fontsize,
                path_effects=[mpe.withStroke(linewidth=linewidth,
                                             foreground=PAPER)])


# ---------------------------------------------------------------------------
# 10. STATISTICS
# ---------------------------------------------------------------------------
def wilson_ci(k, n, z=1.959963984540054):
    """Wilson score interval for a binomial proportion. Closed-form and
    deterministic: it needs the two counts and nothing else, which is why it is
    the only interval this set is allowed to add to a figure that does not
    already carry one."""
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    denom = 1 + z**2 / n
    centre = p + z**2 / (2 * n)
    adj = z * ((p * (1 - p) / n + z**2 / (4 * n**2)) ** 0.5)
    # Clamped to [0, 1] and to the estimate itself. At k = 0 the exact lower
    # bound is 0 and the arithmetic lands a few parts in 10^18 either side of
    # it, which is not a real interval but is enough to make a plotting library
    # refuse a negative error bar.
    lo = min(p, max(0.0, (centre - adj) / denom))
    hi = max(p, min(1.0, (centre + adj) / denom))
    return (lo, hi)


# Notebook-local bindings the drawing cells below expect. They exist in the
# scripts as module globals and are reproduced here so a figure cell is the
# same code in both places.
import statistics as st, glob
ROOT = pathlib.Path(REPO)
HERE = pathlib.Path(FIGDIR)
SUPP = pathlib.Path(RESULTS_ACCORD) / "decontam"
MANIFEST = []
man = {}


def load(*parts):
    path = os.path.join(RESULTS_ACCORD, *parts)
    with open(path) as f:
        return json.load(f), path


def register(fname, label, sources, note=""):
    MANIFEST.append({"file": os.path.join("figures", fname), "label": label,
                     "sources": sources, "note": note})


def mcc_from_counts(tp, tn, fp, fn):
    num = tp * tn - fp * fn
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return 0.0 if den == 0 else num / den
# <<< generated from style.py


## 4. The data these figures are built from

Every figure below reads its numbers out of a stored result file. Nothing is
typed in from a draft, a note or from memory. This cell lists each file, checks
it is present, and shows its top-level keys, so the source material is visible
before a single figure is drawn.

If a file is missing here, the figure that depends on it will fail loudly rather
than fall back to a literal.

In [4]:
SOURCES = [
    ("results/accord/exemplar/exemplar_correctness_recomputed.json", "Fig 1, 4"),
    ("results/accord/diag/step0_1_join_and_calibration.json",        "Fig 1, 2, 5"),
    ("results/accord/diag/step2_signal_check.json",                  "Fig 1, 3, 5"),
    ("results/accord/decontam/corrected_gold_metrics.json",          "Fig 1, 8, 10"),
    ("results/accord/reconciliation_c28.json",                       "Fig 2"),
    ("results/accord/stats/cohort_icd_counts.json",                  "Fig 3"),
    ("results/accord/generality/transfer_results_correct.json",      "Fig 3, 6"),
    ("results/accord/VERIFIED_FACTS.json",                           "Fig 3"),
    ("results/accord/prevalence_replication/sweep_results.json",     "Fig 5"),
    ("results/accord/decontam/accord_v2_decontam_gold_aggregate.json","Fig 6"),
    ("results/accord/section6_auroc_power.json",                     "Fig 7"),
    ("results/task_runs.json",                                       "Fig 7"),
    ("results/accord/stats/intervals.json",                          "Fig 8"),
    ("results/accord/survey/coding_summary.json",                    "Fig 9"),
    ("results/accord/control/final_report.json",                     "Fig 8, 10"),
    ("results/accord/scaling/scaling_results.json",                  "Fig 11"),
    ("results/accord/selection/threshold_robustness_results.json",   "Fig 12"),
    ("results/accord/diag/step3_baselines.json",                     "Fig 14"),
    ("results/accord/differential/exp1a_gold_error_rates_by_group.json", "Fig 2, 12"),
    ("results/accord/number_provenance_audit.json",                  "provenance record"),
]

print(f"{'file':66s} {'kB':>6}  used by")
print("-" * 100)
missing = []
for rel, used in SOURCES:
    path = os.path.join(REPO, rel)
    if not os.path.exists(path):
        print(f"{rel:66s} {'--':>6}  MISSING")
        missing.append(rel)
        continue
    kb = os.path.getsize(path) / 1024
    print(f"{rel:66s} {kb:6.1f}  {used}")

print()
if missing:
    print(f"{len(missing)} of {len(SOURCES)} files are missing. The figures that read them will")
    print("fail rather than substitute a value. Check that the Drive bundle copied correctly.")
else:
    print(f"all {len(SOURCES)} source files present")

print()
print("top-level keys of each file")
print("-" * 100)
for rel, _ in SOURCES:
    path = os.path.join(REPO, rel)
    if not os.path.exists(path):
        continue
    try:
        d = json.load(open(path))
    except Exception as e:
        print(f"{rel}: could not parse ({e})")
        continue
    keys = list(d.keys()) if isinstance(d, dict) else [f"(list of {len(d)})"]
    shown = ", ".join(str(k) for k in keys[:6])
    more = f"  ... and {len(keys) - 6} more" if len(keys) > 6 else ""
    print(f"{rel.split('/')[-1]:52s} {shown}{more}")


file                                                                   kB  used by
----------------------------------------------------------------------------------------------------
results/accord/exemplar/exemplar_correctness_recomputed.json          1.4  Fig 1, 4
results/accord/diag/step0_1_join_and_calibration.json                 4.2  Fig 1, 2, 5
results/accord/diag/step2_signal_check.json                           2.0  Fig 1, 3, 5
results/accord/decontam/corrected_gold_metrics.json                   1.1  Fig 1, 8, 10
results/accord/reconciliation_c28.json                               12.8  Fig 2
results/accord/stats/cohort_icd_counts.json                           0.4  Fig 3
results/accord/generality/transfer_results_correct.json               0.9  Fig 3, 6
results/accord/VERIFIED_FACTS.json                                  220.3  Fig 3
results/accord/prevalence_replication/sweep_results.json             16.8  Fig 5
results/accord/decontam/accord_v2_decontam_gold_aggregate.json

## Figure 1: `fig-pipeline-status`


In [5]:
p_exemplar = os.path.join(RESULTS_ACCORD, "exemplar", "exemplar_correctness_recomputed.json")
d_exemplar = json.load(open(p_exemplar))
ex = d_exemplar["rule_1_2_pooled"]
defect_rate = ex["n_wrong"] / ex["n_resolved"]

p_step01 = os.path.join(RESULTS_ACCORD, "diag", "step0_1_join_and_calibration.json")
d_step01 = json.load(open(p_step01))
xtab = d_step01["weak_confidence_is_degenerate"]["crosstab_label_x_confidence"]
weak_pos = xtab["0.95"]["contradiction"]
weak_neg = xtab["1.0"]["not_contradiction"]
weak_pos_rate = weak_pos / (weak_pos + weak_neg)

p_step2 = os.path.join(RESULTS_ACCORD, "diag", "step2_signal_check.json")
d_step2 = json.load(open(p_step2))
pharm_rate_all = d_step2["contra_prob_vs_pharm_label__ambiguous_as_negative_n299"]["base_rate"]

# One multiplier for the whole thesis, and it is the gold-set one: the weak
# labeller's marginal on the 299 annotated pairs over the pharmacist's base
# rate on those same 299 pairs. Both terms are measured on the same rows, which
# is the only version of this ratio that compares like with like. This figure
# used to divide the training-pool marginal by the gold base rate, a different
# denominator on each side of the division, which gave a second slightly
# different headline number that the introduction then spent a sentence in two
# captions explaining away.
p_exp1a = os.path.join(RESULTS_ACCORD, "differential", "exp1a_gold_error_rates_by_group.json")
d_exp1a = json.load(open(p_exp1a))
gold_weak_pos = sum(g["TP"] + g["FP"] for g in d_exp1a["language_group3"])
gold_n = sum(g["n"] for g in d_exp1a["language_group3"])
ratio = (gold_weak_pos / gold_n) / pharm_rate_all

p_decontam = os.path.join(RESULTS_ACCORD, "decontam", "corrected_gold_metrics.json")
d_decontam = json.load(open(p_decontam))
auroc = d_decontam["aggregate"]["ensemble_auroc"]
ci = d_decontam["aggregate"]["ensemble_ci95"]

print("fig-pipeline-status: values plotted")
print(f"  exemplar defect rate, rule_1_2_pooled     : {ex['n_wrong']}/{ex['n_resolved']} = {defect_rate:.4f}")
print(f"  weak-label positive rate, train pool       : {weak_pos}/{weak_pos + weak_neg} = {weak_pos_rate:.4f}")
print(f"  weak-label positive rate, gold set          : {gold_weak_pos}/{gold_n} = {gold_weak_pos / gold_n:.4f}")
print(f"  pharmacist base rate, n=299 (ambig=neg)     : {pharm_rate_all:.4f}")
print(f"  prevalence mismatch ratio, gold set on gold set: {ratio:.1f}x")
print(f"  decontaminated 3-seed ensemble AUROC         : {auroc:.4f}, 95% CI [{ci[0]:.3f}, {ci[1]:.3f}]")
print("  two categorical, unsourced-by-number badges (stage 'Detector training'): see markdown above")

# Six evenly spaced columns. Every string below is wrapped to at most about
# twelve characters so that no badge and no diagnostic can grow wider than
# its column: the previous version wrapped to the natural phrase length and
# the long badges collided with their neighbours and with the row beneath.

fig-pipeline-status: values plotted
  exemplar defect rate, rule_1_2_pooled     : 65/137 = 0.4745
  weak-label positive rate, train pool       : 1425/1679 = 0.8487
  weak-label positive rate, gold set          : 261/299 = 0.8729
  pharmacist base rate, n=299 (ambig=neg)     : 0.0401
  prevalence mismatch ratio, gold set on gold set: 21.8x
  decontaminated 3-seed ensemble AUROC         : 0.5145, 95% CI [0.321, 0.703]
  two categorical, unsourced-by-number badges (stage 'Detector training'): see markdown above


**Step 2. The figure.** Drawn only from the values printed above, then written to `fig-pipeline-status.pdf`.

In [ ]:
exemplar, p_exemplar = load("exemplar", "exemplar_correctness_recomputed.json")
# The former reference rate was supplied by a task description, never measured
# (REVISION_LOG R8). Use the measured wrong rate on the resolved candidates.
_ex = exemplar["rule_1_2_pooled"]
defect_rate = _ex["n_wrong"] / _ex["n_resolved"]

step01, p_step01 = load("diag", "step0_1_join_and_calibration.json")
xtab = step01["weak_confidence_is_degenerate"]["crosstab_label_x_confidence"]
weak_pos = xtab["0.95"]["contradiction"]
weak_neg = xtab["1.0"]["not_contradiction"]
weak_pos_rate = weak_pos / (weak_pos + weak_neg)

step2, p_step2 = load("diag", "step2_signal_check.json")
pharm_rate = step2["contra_prob_vs_pharm_label__ambiguous_excluded_n253"]["base_rate"]
# One multiplier for the whole thesis, and it is the gold-set one: the weak
# labeller's marginal on the 299 annotated pairs over the pharmacist's base
# rate on those same 299 pairs. Both terms are measured on the same rows,
# which is the only version of this ratio that compares like with like.
# This figure used to divide the training-pool marginal by the gold base
# rate, which is a different denominator on each side of the division and
# gave a second, slightly different headline number that the introduction
# then had to spend a sentence in two captions explaining away.
exp1a, p_exp1a = load("differential", "exp1a_gold_error_rates_by_group.json")
_lang = exp1a["language_group3"]
gold_weak_pos = sum(g["TP"] + g["FP"] for g in _lang)
gold_n = sum(g["n"] for g in _lang)
_d299 = step2["contra_prob_vs_pharm_label__ambiguous_as_negative_n299"]
pharm_rate_all = _d299["base_rate"]
ratio = (gold_weak_pos / gold_n) / pharm_rate_all

decontam, p_decontam = load("decontam", "corrected_gold_metrics.json")
auroc = decontam["aggregate"]["ensemble_auroc"]
ci = decontam["aggregate"]["ensemble_ci95"]

# Six full-canvas-width row bands, no box column and no connectors: reading
# order down the page plus the index column carries the sequence, and six
# arrows on a band chart is noise. Drawn on a strict point grid, one data
# unit to one typographic point, so the bands are identical by construction.
W, H = FIGW, 3.10
fig, ax = plt.subplots(figsize=(W, H))
canvas(fig)
plate(fig)
fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
ax.set_xlim(0, W * 72)
ax.set_ylim(0, H * 72)
ax.axis("off")

# (stage, verdict word, numeric detail, diagnostic, state)
PASS, NONE, FAULT = "pass", "none", "fault"
rows = [
    ("Cohort construction", "Works", "",
     "cohort counts\nreconcile", PASS),
    ("Medication extraction", "Unvalidated", "",
     "no diagnostic run", NONE),
    ("Rule-based candidate\ngeneration", "Fails",
     f"{defect_rate*100:.1f}% wrong of\nresolved candidates",
     "action-set match vs\nposition index", FAULT),
    ("LLM weak supervision", "Fails",
     f"{ratio:.0f}x prevalence\nmismatch, gold set",
     "weak-label marginal\nvs gold marginal", FAULT),
    ("Detector training", "Falsified", "three claims",
     r"$\|\Delta\theta\|$ from init," + "\n" + r"OOD transfer $F_1$", FAULT),
    ("Evaluation", "Fails",
     f"AUROC {auroc:.3f},\n95% CI [{ci[0]:.2f}, {ci[1]:.2f}]",
     "bootstrap CI on\nAUROC before write-up", FAULT),
]
# Band fill F1, chip fill F2. The no-diagnostic chip is MUTE rather than the
# SLATE its band stroke uses: white text on SLATE is 3.48:1, under this
# set's own floor, and on MUTE it is 4.83:1. The band still reads as the
# quiet state because the band, not the chip, carries the area.
STATE = {
    PASS:  (EMERALD_TINT, EMERALD_DEEP, (EMERALD_DEEP, EMERALD_CORE)),
    NONE:  (MIST, SLATE, (MUTE, SLATE)),
    FAULT: (SCARLET_TINT, SCARLET_DEEP, (SCARLET_DEEP, SCARLET_CORE)),
}

x_idx, x_name = 13.0, 24.0
x_chip, w_chip = 148.0, 58.0
x_detail, x_diag = 210.0, 294.0
band_h, gutter = 29.4, 3.0
y_top = H * 72 - 30.0
for i, (name, verdict, detail, diag, state) in enumerate(rows):
    tint, deep, chip_hue = STATE[state]
    y = y_top - (i + 1) * band_h - i * gutter
    mid = y + band_h / 2
    ax.add_patch(FancyBboxPatch(
        (1.5, y), W * 72 - 3.0, band_h,
        boxstyle="round,pad=0,rounding_size=2.0", mutation_aspect=1.0,
        linewidth=LW_NODE, edgecolor=deep, facecolor=tint, zorder=2))
    ax.text(x_idx, mid, str(i + 1), ha="center", va="center",
            fontsize=10.0, color=deep, zorder=5)
    ax.text(x_name, mid, name, ha="left", va="center", fontsize=10.5,
            color=INK, linespacing=1.18, zorder=5)
    chip(ax, x_chip, mid - 8.0, w_chip, 16.0, chip_hue, verdict,
         fontsize=T_CHIP, radius=3.0, zorder=5)
    if detail:
        ax.text(x_detail, mid, detail, ha="left", va="center",
                fontsize=T_SERIES, color=INK, linespacing=1.22, zorder=5)
    ax.text(x_diag, mid, diag, ha="left", va="center", fontsize=T_NOTE,
            style="italic", color=MUTE, linespacing=1.22, zorder=5)

# 335.6pt at 13pt against 380pt of inner plate: the longer wording ran to
# 381.8pt and set its terminal period on the plate's rounded right corner.
signature(fig, SCARLET_DEEP,
          "Five of six stages failed the diagnostic run on them, "
          "or had none.")
fname = "fig-pipeline-status.pdf"
fig.savefig(os.path.join(FIGDIR, fname))
plt.close(fig)
register(fname, "fig:pipeline-status",
         [p_exemplar, p_step01, p_step2, p_decontam],
         "Two stage verdicts are categorical narrative with no committed numeric source "
         "(CORRECTIONS.md). The three parenthetical numbers "
         "(47.45% wrong of resolved, ~22x, AUROC 0.5145 with CI) are computed from the cited JSON files. "
         "The multiplier is the gold-set one, 261/299 over 12/299, the same ratio "
         "fig:intro-headline brackets. "
         "AUROC corrected per CORRECTIONS.md C16 (was 0.460, now the decontaminated "
         "3-seed ensemble 0.5145).")


## Figure 2: `fig-intro-headline`


In [7]:
p_step01 = os.path.join(RESULTS_ACCORD, "diag", "step0_1_join_and_calibration.json")
d_step01 = json.load(open(p_step01))
xtab = d_step01["weak_confidence_is_degenerate"]["crosstab_label_x_confidence"]
weak_pos, weak_neg = xtab["0.95"]["contradiction"], xtab["1.0"]["not_contradiction"]
weak_n = weak_pos + weak_neg
weak_rate = weak_pos / weak_n

p_exp1a = os.path.join(RESULTS_ACCORD, "differential", "exp1a_gold_error_rates_by_group.json")
d_exp1a = json.load(open(p_exp1a))
lang = d_exp1a["language_group3"]
pred_pos = sum(g["TP"] + g["FP"] for g in lang)
pred_n = sum(g["n"] for g in lang)
pred_rate = pred_pos / pred_n

p_step2 = os.path.join(RESULTS_ACCORD, "diag", "step2_signal_check.json")
d_step2 = json.load(open(p_step2))
d299 = d_step2["contra_prob_vs_pharm_label__ambiguous_as_negative_n299"]
pharm_pos, pharm_n = d299["n_pos"], d299["n"]
pharm_rate = pharm_pos / pharm_n

p_intervals = os.path.join(RESULTS_ACCORD, "stats", "intervals.json")
d_intervals = json.load(open(p_intervals))
mcc = d_intervals["E2_mcc_intervals"]
order = ["gpt4o_zeroshot", "gpt4o_fewshot", "accord_binary_at_0.5_253core", "llama_baseline"]
labels = ["GPT-4o zero-shot", "GPT-4o few-shot", "ACCORD (contam.)", "Llama 3 8B"]

p_c28 = os.path.join(RESULTS_ACCORD, "reconciliation_c28.json")
d_c28 = json.load(open(p_c28))
accord_pred = d_c28["a_what_each_number_counts"]["278"]
weak_marginal = d_c28["a_what_each_number_counts"]["261"]

def wilson_ci(k, n, z=1.959963984540054):
    """Wilson score interval for a binomial proportion; closed-form, deterministic."""
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    denom = 1 + z ** 2 / n
    centre = p + z ** 2 / (2 * n)
    adj = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2))
    # Clamped to [0, 1] and to the estimate itself, identically to the helper in
    # the setup cell: at k = 0 the exact lower bound is 0 and the arithmetic
    # lands a few parts in 10^18 either side of it.
    lo = min(p, max(0.0, (centre - adj) / denom))
    hi = max(p, min(1.0, (centre + adj) / denom))
    return (lo, hi)

print("fig-intro-headline: values plotted")
print(f"  weak-label positive rate, train pool     : {weak_pos}/{weak_n} = {weak_rate:.4f}")
print(f"  left-panel middle bar (exp1a TP+FP sum)  : {pred_pos}/{pred_n} = {pred_rate:.4f}  (the weak labeller on the gold set, NOT ACCORD)")
print(f"  pharmacist true rate, gold set (n=299)   : {pharm_pos}/{pharm_n} = {pharm_rate:.4f}")
for k in order:
    v = mcc[k]
    print(f"  MCC {k:32s}: {v['mcc']:+.4f}  95% CI [{v['ci95'][0]:+.4f}, {v['ci95'][1]:+.4f}]")
print(f"  reconciliation_c28: ACCORD predicted-positive = {accord_pred['value']}/{accord_pred['denominator']}"
      f"  (accord_pred == 'contradiction')")
print(f"  reconciliation_c28: weak-labeller marginal    = {weak_marginal['value']}/{weak_marginal['denominator']}"
      f"  (weak_label == 'contradiction', matches the {pred_pos}/{pred_n} bar above)")

# The right panel gets the wider share: its four category labels need room
# side by side, while the left panel's y labels live outside its axes.

fig-intro-headline: values plotted
  weak-label positive rate, train pool     : 1425/1679 = 0.8487
  left-panel middle bar (exp1a TP+FP sum)  : 261/299 = 0.8729  (the weak labeller on the gold set, NOT ACCORD)
  pharmacist true rate, gold set (n=299)   : 12/299 = 0.0401
  MCC gpt4o_zeroshot                  : +0.1089  95% CI [-0.0319, +0.2484]
  MCC gpt4o_fewshot                   : +0.0117  95% CI [-0.1120, +0.1319]
  MCC accord_binary_at_0.5_253core    : -0.0070  95% CI [-0.1626, +0.0753]
  MCC llama_baseline                  : -0.0436  95% CI [-0.2337, +0.0563]
  reconciliation_c28: ACCORD predicted-positive = 278/299  (accord_pred == 'contradiction')
  reconciliation_c28: weak-labeller marginal    = 261/299  (weak_label == 'contradiction', matches the 261/299 bar above)


**Step 2. The figure.** Drawn only from the values printed above, then written to `fig-intro-headline.pdf`.

In [ ]:
step01, p_step01 = load("diag", "step0_1_join_and_calibration.json")
xtab = step01["weak_confidence_is_degenerate"]["crosstab_label_x_confidence"]
weak_pos, weak_neg = xtab["0.95"]["contradiction"], xtab["1.0"]["not_contradiction"]
weak_n = weak_pos + weak_neg
weak_rate = weak_pos / weak_n

exp1a, p_exp1a = load("differential", "exp1a_gold_error_rates_by_group.json")
lang = exp1a["language_group3"]
pred_pos = sum(g["TP"] + g["FP"] for g in lang)
pred_n = sum(g["n"] for g in lang)
pred_rate = pred_pos / pred_n

step2, p_step2 = load("diag", "step2_signal_check.json")
d = step2["contra_prob_vs_pharm_label__ambiguous_as_negative_n299"]
pharm_pos, pharm_n = d["n_pos"], d["n"]
pharm_rate = pharm_pos / pharm_n

# Panel (b) is gone. Its four Matthews-correlation rows are Figure 8's
# first family, drawn identically 85 pages later, and Figure 8 is now the
# one place every headline estimate in this thesis lives. What remains is
# the number the rest of the thesis follows: a rate of a whole, so bars.
fig, ax = plt.subplots(figsize=(FIGW, 2.55))
canvas(fig)
# top was 0.735, which left 11pt of dead paper between the headline and the
# frame. The sub-line is now one line by directive, so 250pt of scarlet type
# has left the white right-hand third of the plot; the axes takes back the
# dead band instead, which is where that ink should have come from anyway.
# 6.8pt of clearance remains under the headline's second line.
fig.subplots_adjust(left=0.300, right=0.985, top=0.752, bottom=0.170)

bars = [("Weak labeller, train pool", weak_pos, weak_n, wilson_ci(weak_pos, weak_n)),
        ("Weak labeller, gold set", pred_pos, pred_n, wilson_ci(pred_pos, pred_n)),
        ("Pharmacist truth, gold set", pharm_pos, pharm_n, wilson_ci(pharm_pos, pharm_n))]
ypos = np.arange(len(bars))[::-1]
rates = [b[1] / b[2] for b in bars]
los = [b[3][0] for b in bars]
his = [b[3][1] for b in bars]
errs = [[r - lo for r, lo in zip(rates, los)],
        [hi - r for r, hi in zip(rates, his)]]
# The weak labeller is Llama 3 8B (Chapter 3), so it carries Llama's hue on
# both of its rows; the pharmacist carries the truth hue.
hues = [PLUM, PLUM, EMERALD]

ax.set_xlim(0, 1.86)
ax.set_ylim(-0.66, 2.66)
# The gap between the pharmacist's rate and the weak labeller's on the same
# 299 pairs IS the finding, so it is drawn as a region rather than left for
# the reader to measure off the axis.
band(ax, pharm_pos / pharm_n, pred_pos / pred_n, SCARLET_WASH, zorder=0)
ax.barh(ypos, rates, color=[h.CORE for h in hues], height=0.62,
        linewidth=LW_BAR, edgecolor=[h.DEEP for h in hues], zorder=3)
# A same-colour interval loses its lower half inside the fill, so the
# estimate reads one-sided and the bar reads as if it were exact. Casing is
# what keeps both halves readable.
ax.errorbar(rates, ypos, xerr=errs, fmt="none", ecolor=PAPER,
            elinewidth=LW_INTERVAL + 1.4, capthick=LW_INTERVAL + 1.4,
            capsize=ERRBAR["capsize"], zorder=4)
for y, r, e_lo, e_hi, hue in zip(ypos, rates, errs[0], errs[1], hues):
    ax.errorbar([r], [y], xerr=[[e_lo], [e_hi]], fmt="none",
                ecolor=hue.CORE, zorder=5, **ERRBAR)
for y, b, r, hi, hue in zip(ypos, bars, rates, his, hues):
    # The rate at the bar, the denominators in the row label: two
    # different jobs, and side by side they collided with the bracket.
    ax.text(hi + 0.030, y, f"{r*100:.1f}%", va="center", ha="left",
            fontsize=T_SERIES, color=hue.DEEP, zorder=6)
ax.set_yticks(ypos)
ax.set_yticklabels([f"{b[0]}\n({b[1]:,} of {b[2]:,})" for b in bars],
                   fontsize=T_SERIES, linespacing=1.3)
ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
ax.set_xlabel("Positive rate")
frame(ax)

# The claim numeral. The bracket spans only the two bars measured on the
# same 299-pair gold set; the training-pool bar has a different denominator
# and is not part of the comparison.
gold_ratio = (pred_pos / pred_n) / (pharm_pos / pharm_n)
x_br = 1.14
ax.plot([x_br] * 2, [ypos[2], ypos[1]], color=SCARLET_CORE, lw=LW_DATA,
        solid_capstyle="butt", zorder=6)
for y in (ypos[1], ypos[2]):
    ax.plot([x_br, x_br - 0.045], [y, y], color=SCARLET_CORE, lw=LW_DATA,
            zorder=6)
# The sub-line is ONE line, and one line of it is 167pt: it does not fit in
# the 94pt between the bracket and the right spine, so it goes BELOW the
# bracket, flush to the right spine, and runs left across the empty row
# under the pharmacist bar. Set beside the bracket it wrapped to four lines
# and the fourth was struck through by the bottom spine. Its left third
# crosses the scarlet wash; SCARLET_DEEP on SCARLET_WASH is 6.29:1.
ax.text(x_br + 0.06, (ypos[1] + ypos[2]) / 2, f"{gold_ratio:.0f}x",
        va="center", ha="left", fontsize=T_CLAIM, fontweight="bold",
        color=SCARLET_DEEP, zorder=6)
ax.text(1.817, -0.24,
        f"the weak labeller's marginal over the pharmacist's, "
        f"same {pharm_n} pairs",
        va="top", ha="right", fontsize=T_SERIES, color=SCARLET_DEEP,
        zorder=6)

signature(fig, PLUM_DEEP,
          f"The weak labeller calls {pred_pos/pred_n*100:.0f}% of gold-set "
          f"pairs contradictions.\n"
          f"The pharmacist calls {pharm_pos/pharm_n*100:.0f}%.")
fname = "fig-intro-headline.pdf"
fig.savefig(os.path.join(FIGDIR, fname))
plt.close(fig)
register(fname, "fig:intro-headline",
         [p_step01, p_exp1a, p_step2],
         "The left-panel middle bar is the weak labeller marginal, 261 of 299 "
         "(gold.weaklabel_contradiction); ACCORD predicts contradiction for 278 of 299 "
         "(gold.accord_pred_contradiction, reconciliation_c28.json). The Matthews "
         "correlation panel that used to sit under this one is deleted: those four "
         "estimates are fig:forest's first family.")


## Figure 3: `fig-cohort-flow`


In [9]:
p_icd = os.path.join(RESULTS_ACCORD, "stats", "cohort_icd_counts.json")
hf = json.load(open(p_icd))["heart_failure"]

p_transfer = os.path.join(RESULTS_ACCORD, "generality", "transfer_results_correct.json")
n_candidates = json.load(open(p_transfer))["hf_reference"]["n_candidates"]

p_step2 = os.path.join(RESULTS_ACCORD, "diag", "step2_signal_check.json")
d_step2 = json.load(open(p_step2))
d299 = d_step2["contra_prob_vs_pharm_label__ambiguous_as_negative_n299"]
d253 = d_step2["contra_prob_vs_pharm_label__ambiguous_excluded_n253"]
d_amb = d_step2["contra_prob_vs_pharm_label__ambiguous_as_positive_n299"]
# The ambiguous count is not stored on its own line. It is the difference
# between the two positive counts recorded in the same file on the same 299
# rows, ambiguous counted positive against ambiguous counted negative, which is
# the same subtraction the prevalence figure performs.
n_ambiguous = d_amb["n_pos"] - d299["n_pos"]

# Four stages have no primary artifact inside this repository: their upstream
# summaries live outside it under the data agreement. Their values are read from
# the thesis register, which is committed here and records the artifact each one
# came from.
p_facts = os.path.join(RESULTS_ACCORD, "VERIFIED_FACTS.json")
facts = json.load(open(p_facts))
fact = lambda key: facts[key]["value"]

n_both_roles = fact("cohort_chain.has_both_roles")
n_pairs = fact("cohort_chain.valid_admissions_both_selected")
n_note_rows = fact("cohort_chain.total_note_pair_rows")
n_mentions = fact("extraction.medication_extraction.total_mentions")
n_classified = fact("extraction.medication_extraction.classified_mentions")
n_unclassified = fact("extraction.medication_extraction.unclassified_derived")
n_drawn = fact("extraction.annotation_sample.total_sampled")

print("fig-cohort-flow: values plotted")
print(f"  ICD prefix admissions      {hf['pre_los']:>8,}")
print(f"  removed by length of stay  {hf['removed']:>8,}")
print(f"  after length of stay       {hf['post_los']:>8,}")
print(f"  both note roles present    {n_both_roles:>8,}")
print(f"  note pairs selected        {n_pairs:>8,}   note rows {n_note_rows:,}")
print(f"  medication mentions        {n_mentions:>8,}")
print(f"  no action class            {n_unclassified:>8,}")
print(f"  action classified          {n_classified:>8,}")
print(f"  candidates after prefilter {n_candidates:>8,}")
print(f"  annotated pairs            {d299['n']:>8,}   of {n_drawn:,} drawn")
print(f"  coded ambiguous            {n_ambiguous:>8,}")
print(f"  non-ambiguous core         {d253['n']:>8,}   positives {d253['n_pos']}")


fig-cohort-flow: values plotted
  ICD prefix admissions        13,608
  removed by length of stay        11
  after length of stay         13,597
  both note roles present       9,820
  note pairs selected           9,783   note rows 19,566
  medication mentions         219,683
  no action class             131,347
  action classified            88,336
  candidates after prefilter    1,978
  annotated pairs                 299   of 300 drawn
  coded ambiguous                  46
  non-ambiguous core              253   positives 12


**Step 2. The figure.** Drawn only from the values printed above, then written to `fig-cohort-flow.pdf`.

In [ ]:
icd, p_icd = load("stats", "cohort_icd_counts.json")
hf = icd["heart_failure"]

transfer, p_transfer = load("generality", "transfer_results_correct.json")
n_candidates = transfer["hf_reference"]["n_candidates"]

step2, p_step2 = load("diag", "step2_signal_check.json")
d299 = step2["contra_prob_vs_pharm_label__ambiguous_as_negative_n299"]
d253 = step2["contra_prob_vs_pharm_label__ambiguous_excluded_n253"]
d_amb = step2["contra_prob_vs_pharm_label__ambiguous_as_positive_n299"]
# The ambiguous count is not stored on its own line. It is the difference
# between the two positive counts recorded in the same file on the same 299
# rows, ambiguous counted positive against ambiguous counted negative, which
# is the same subtraction make_fig_prevalence already performs.
n_ambiguous = d_amb["n_pos"] - d299["n_pos"]

facts, p_facts = load("VERIFIED_FACTS.json")
def fact(key):
    return facts[key]["value"]

n_both_roles = fact("cohort_chain.has_both_roles")
n_pairs = fact("cohort_chain.valid_admissions_both_selected")
n_note_rows = fact("cohort_chain.total_note_pair_rows")
n_mentions = fact("extraction.medication_extraction.total_mentions")
n_classified = fact("extraction.medication_extraction.classified_mentions")
n_unclassified = fact("extraction.medication_extraction.unclassified_derived")
n_drawn = fact("extraction.annotation_sample.total_sampled")

# A true branching flow in three phase groups. The phase band is a WASH
# ground and the stage boxes inside it are the TINT of the same hue, so the
# group reads as a group and the boxes still read as boxes; a band and its
# boxes in one stop would differ only by their strokes. One data unit is
# one typographic point.
W, H = W80, 3.80
fig, ax = plt.subplots(figsize=(W, H))
canvas(fig)
plate(fig)
fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
ax.set_xlim(0, W * 72)
ax.set_ylim(0, H * 72)
ax.axis("off")
x_right = W * 72 - 1.5

# (stage name, count, exclusion below this stage or None)
stages = [
    ("Heart failure admissions, ICD-9 prefix", f"{hf['pre_los']:,}",
     f"{hf['removed']:,} admissions with no\npositive length of stay"),
    ("After the length-of-stay filter", f"{hf['post_los']:,}", None),
    ("Admissions carrying both note roles", f"{n_both_roles:,}", None),
    ("Note pair selected, one per role", f"{n_pairs:,} pairs", None),
    ("Medication mentions extracted", f"{n_mentions:,}",
     f"{n_unclassified:,} mentions with no\naction class assigned"),
    ("Mentions carrying an action class", f"{n_classified:,}", None),
    ("Candidates after the temporal prefilter", f"{n_candidates:,}", None),
    ("Pairs annotated by the pharmacist", f"{d299['n']} of {n_drawn}",
     f"{n_ambiguous} pairs coded ambiguous"),
    ("Non-ambiguous evaluation core",
     f"{d253['n']} pairs, {d253['n_pos']} confirmed contradictions", None),
]

# The exclusion stub's arrowhead is 7.4pt long and 7.2pt wide, and it used
# to land with its back edge at x = 225.6 against stage boxes that ended at
# 228: the flare fused with the rounded bottom-right and top-right corners
# of the boxes above and below, and both strokes were interrupted. The head
# moves clear of the boxes instead of the boxes shrinking, because the two
# chromatic boxes on this canvas carry its whole chroma budget. The stage
# column shifts 2pt left, the gutter 4pt right, and the head now stops 3pt
# short of the gutter: back edge 231.6, box stroke outer edge 226.6, 5.0pt
# of clear paper between them. The gutter node loses 2pt of the 8pt it had
# spare on its longest line.
box_x, box_w = 24.0, 202.0
x_gut = 242.0
stub_gap = 3.0
box_h, last_h, gap = 17.0, 28.0, 8.0
y_top = H * 72 - 44.0
tops, y = [], y_top
for i in range(len(stages)):
    h = last_h if i == len(stages) - 1 else box_h
    tops.append((y, h))
    y -= h + gap

# Phase groups: rows, WASH band, box TINT, DEEP stroke, rail, name.
phases = [
    (range(0, 4), HAIR, MIST, SLATE, SLATE, "cohort construction"),
    (range(4, 7), COBALT_WASH, COBALT_TINT, COBALT_DEEP, COBALT_DEEP,
     "extraction and mining"),
    (range(7, 9), EMERALD_WASH, EMERALD_TINT, EMERALD_DEEP, EMERALD_DEEP,
     "annotation"),
]
phase_of = {}
for rows, wash, tint, deep, rail, name in phases:
    idx = list(rows)
    for i in idx:
        phase_of[i] = (tint, deep)
    b_hi = tops[idx[0]][0] + 5.0
    b_lo = tops[idx[-1]][0] - tops[idx[-1]][1] - 5.0
    ax.add_patch(mpatches.Rectangle((1.5, b_lo), x_right - 1.5, b_hi - b_lo,
                                    facecolor=wash, edgecolor="none",
                                    linewidth=0, zorder=0))
    ax.add_patch(mpatches.Rectangle((7.0, b_lo + 2.0), 3.0,
                                    b_hi - b_lo - 4.0, facecolor=rail,
                                    edgecolor="none", linewidth=0, zorder=1))
    ax.text(17.0, (b_lo + b_hi) / 2, name, ha="center", va="center",
            rotation=90, fontsize=T_NOTE, color=MUTE, zorder=2)

x_stem = box_x + box_w / 2
for i, ((t, h), (name, count, excl)) in enumerate(zip(tops, stages)):
    last = i == len(stages) - 1
    mid = t - h / 2
    if last:
        # The evaluated core: the one solid element, and the end of the path.
        chip(ax, box_x, t - h, box_w, h, SCARLET,
             f"{name}\n{count}", fontsize=T_SERIES, radius=3.0, zorder=4)
    else:
        tint, deep = phase_of[i]
        node(ax, box_x, t - h, box_w, h, tint, deep, radius=2.5, zorder=4)
        ax.text(box_x + 7.0, mid, name, ha="left", va="center",
                fontsize=T_SERIES, color=INK, zorder=5)
        ax.text(box_x + box_w - 7.0, mid, count, ha="right", va="center",
                fontsize=T_AXIS, color=deep, zorder=5)
    if not last:
        arrow(ax, (x_stem, t - h), (x_stem, t - h - gap), color=GRAPHITE,
              lw=LW_CONNECTOR, zorder=6)
    if excl is not None:
        # An exclusion is accounting, not failure: SLATE and MIST, never
        # scarlet. It leaves the trunk in the gap between two stages, so it
        # reads as taken out between them.
        y_gap = t - h - gap / 2
        arrow(ax, (x_stem + 4.0, y_gap), (x_gut - stub_gap, y_gap),
              color=SLATE, lw=LW_CONNECTOR, zorder=6)
        lines = excl.count("\n") + 1
        ch = 11.0 * lines + 7.0
        node(ax, x_gut, y_gap - ch / 2, x_right - 4.0 - x_gut, ch, MIST,
             SLATE, radius=2.5, zorder=4)
        ax.text(x_gut + 6.0, y_gap, excl, ha="left", va="center",
                fontsize=T_NOTE, color=GRAPHITE, linespacing=1.24, zorder=5)

ax.text(x_gut, y_top + 8.0, "excluded", ha="left", va="bottom",
        fontsize=T_NOTE, color=MUTE, style="italic")

signature(fig, SCARLET_DEEP,
          "13,608 admissions in; 253 pairs and\n"
          "12 confirmed contradictions out.")
fname = "fig-cohort-flow.pdf"
fig.savefig(os.path.join(FIGDIR, fname))
plt.close(fig)
register(fname, "fig:cohort-flow",
         [p_icd, p_transfer, p_step2, p_facts],
         "CONSORT-style path from the source database to the two evaluated sets. Nine "
         "stages, three exclusions, every count read from a committed file. Two exclusion "
         "branches the diagram would otherwise carry are omitted because no committed file "
         "records them as counts: the admissions dropped between the length-of-stay filter "
         "and the both-roles requirement, and those dropped between the both-roles "
         "requirement and the selected pairs. Both are differences of two committed counts "
         "and neither is a recorded exclusion, so neither is drawn. The ambiguous count is "
         "the difference between the two positive counts recorded on the same 299 rows in "
         "the signal-check file, the same subtraction the prevalence figure performs. Four "
         "stages read the thesis register rather than a primary artifact, because their own "
         "upstream summaries live outside this repository under the data agreement.")


## Figure 4: `fig-exemplar`


In [11]:
p_exemplar = os.path.join(RESULTS_ACCORD, "exemplar", "exemplar_correctness_recomputed.json")
d_exemplar = json.load(open(p_exemplar))
ex = d_exemplar["rule_1_2_pooled"]
defect_rate = ex["n_wrong"] / ex["n_resolved"]

print("fig-exemplar: values plotted")
print(f"  rule_1_2_pooled: n_wrong={ex['n_wrong']}, n_resolved={ex['n_resolved']}, defect_rate={defect_rate:.4f}")

# Widened and re-anchored. Previously the trigger caption ran into the
# "Discharge summary" heading, both leader arrows crossed a slot box and struck
# through its label, the right-hand annotations ran off the axes, and the summary
# text was wider than the box drawn around it.

fig-exemplar: values plotted
  rule_1_2_pooled: n_wrong=65, n_resolved=137, defect_rate=0.4745


**Step 2. The figure.** Drawn only from the values printed above, then written to `fig-exemplar.pdf`.

In [ ]:
exemplar, p_exemplar = load("exemplar", "exemplar_correctness_recomputed.json")

# Abstract slots, coloured loudly. No synthetic clinical sentences: a thesis
# about instruments that mislead, built on a MIMIC-III data agreement, does
# not put invented clinical text on a canvas where a reader could take it
# for a record. One data unit is one typographic point.
W, H = W80, 2.45
fig, ax = plt.subplots(figsize=(W, H))
canvas(fig)
plate(fig)
fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
ax.set_xlim(0, W * 72)
ax.set_ylim(0, H * 72)
ax.axis("off")

x_nurse, x_disch, box_w = 58.0, 158.0, 84.0
x_note = 222.0
box_h, pitch = 19.0, 21.8
y_row0 = 106.0
rows_y = [y_row0 - i * pitch for i in range(5)]
x_right = W * 72 - 1.5

# The mismatch is a ROW, so it is drawn as one: a wash ground across the
# whole canvas at the CESS row, under both columns and the annotation.
ax.add_patch(mpatches.Rectangle(
    (1.5, rows_y[4] - box_h / 2 - 4.0), x_right - 1.5, box_h + 8.0,
    facecolor=SCARLET_WASH, edgecolor="none", linewidth=0, zorder=0))

# The trigger bracket over the two column heads: the rule quantifies over
# the whole set, which is the fact the positional selector ignores.
y_br = 133.0
ax.plot([x_nurse, x_disch], [y_br, y_br], color=COBALT_CORE, lw=LW_DATA,
        solid_capstyle="butt", zorder=4)
for x in (x_nurse, x_disch):
    ax.plot([x, x], [y_br - 5.0, y_br], color=COBALT_CORE, lw=LW_DATA,
            solid_capstyle="butt", zorder=4)
ax.text(x_disch + 14.0, y_br, "the rule quantifies over the whole set",
        ha="left", va="center", fontsize=T_SERIES, color=COBALT_DEEP,
        zorder=5)

for x, head in ((x_nurse, "Nursing note"), (x_disch, "Discharge summary")):
    ax.text(x, 120.0, head, ha="center", va="center", fontsize=10.0,
            color=INK, zorder=5)

nurse_tags = ["M1", "M2", "M3", "M4", "M5"]
disch_tags = ["CONT", "CONT", "CONT", "CONT", "CESS"]
for i, y in enumerate(rows_y):
    node(ax, x_nurse - box_w / 2, y - box_h / 2, box_w, box_h,
         COBALT_TINT, COBALT_DEEP, radius=2.5, zorder=3)
    ax.text(x_nurse, y, nurse_tags[i], ha="center", va="center",
            fontsize=T_SERIES, color=INK, zorder=5)
    if i < 4:
        node(ax, x_disch - box_w / 2, y - box_h / 2, box_w, box_h,
             MIST, SLATE, radius=2.5, zorder=3)
        ax.text(x_disch, y, disch_tags[i], ha="center", va="center",
                fontsize=T_SERIES, color=INK, zorder=5)
    else:
        # The one loud element on the canvas.
        chip(ax, x_disch - box_w / 2, y - box_h / 2, box_w, box_h,
             SCARLET, disch_tags[i], fontsize=T_SERIES, radius=2.5,
             zorder=3)

# Annotations align to their own row and are joined to it by a rule on the
# row's own baseline. No diagonals, no leaders.
notes = [
    (0, "selected by position (row 1):\nwhat the annotator, the weak\n"
        "labeller and the model saw", MUTE),
    (4, "the mention the rule\nactually fired on:\naction matched, dose adjacent",
     SCARLET_DEEP),
]
for i, text, tone in notes:
    y = rows_y[i]
    ax.plot([x_disch + box_w / 2 + 4.0, x_note - 5.0], [y, y], color=SLATE,
            lw=1.1, solid_capstyle="butt", zorder=4)
    ax.text(x_note, y, text, ha="left", va="center", fontsize=T_NOTE,
            color=tone, linespacing=1.24, zorder=5)

signature(fig, SCARLET_DEEP,
          "The rule fires on the set.\nThe exemplar shown to everyone is row 1.")
fname = "fig-exemplar.pdf"
fig.savefig(os.path.join(FIGDIR, fname))
plt.close(fig)
register(fname, "fig:exemplar", [p_exemplar],
         "Schematic; slots are abstract labels (M1..M5, CONT/CESS action-tag codes), no "
         "clinical text and no number: the wrong-exemplar rate it used to carry is a "
         "population rate and this panel draws a single candidate, so the rate is stated "
         "in the chapter body instead.")


## Figure 5: `fig-prevalence-response`


In [13]:
p_sweep = os.path.join(RESULTS_ACCORD, "prevalence_replication", "sweep_results.json")
d_sweep = json.load(open(p_sweep))
models = d_sweep["models"]
prompts = ["A_bare", "B_baserate_abstain"]
# The second row label is wrapped: on one line it ran the full height of its
# panel and collided with the row above it.
prompt_label = {"A_bare": "bare prompt", "B_baserate_abstain": "base rate stated,\nabstention allowed"}

# The "clinical" comparison point: same weak-label-vs-pharmacist shift as figures 1 and 2,
# recomputed here from the same two diagnostic files (not typed in).
p_step01 = os.path.join(RESULTS_ACCORD, "diag", "step0_1_join_and_calibration.json")
d_step01 = json.load(open(p_step01))
xtab = d_step01["weak_confidence_is_degenerate"]["crosstab_label_x_confidence"]
weak_pos, weak_neg = xtab["0.95"]["contradiction"], xtab["1.0"]["not_contradiction"]
clinical_y = weak_pos / (weak_pos + weak_neg)

p_step2 = os.path.join(RESULTS_ACCORD, "diag", "step2_signal_check.json")
d_step2 = json.load(open(p_step2))
clinical_x = d_step2["contra_prob_vs_pharm_label__ambiguous_as_negative_n299"]["base_rate"]

print("fig-prevalence-response: values plotted")
print(f"  models: {models}")
print(f"  clinical marker (true prevalence, labeller positive rate): ({clinical_x:.4f}, {clinical_y:.4f})")
for pr in prompts:
    for m in models:
        pts = {p: d_sweep["results"][m][pr].get(p) for p in ["0.02", "0.05", "0.1", "0.25"]}
        rates = {p: (e["positive_rate_of_decided"] if e else None) for p, e in pts.items()}
        print(f"  {m:14s} / {prompt_label[pr]:38s}: {rates}")

fig-prevalence-response: values plotted
  models: ['llama3:8b', 'granite3.3:8b', 'qwen3:8b']
  clinical marker (true prevalence, labeller positive rate): (0.0401, 0.8487)
  llama3:8b      / bare prompt                           : {'0.02': 0.07, '0.05': 0.17, '0.1': 0.06, '0.25': 0.12}
  granite3.3:8b  / bare prompt                           : {'0.02': 0.05, '0.05': 0.07, '0.1': 0.1, '0.25': 0.16}
  qwen3:8b       / bare prompt                           : {'0.02': 0.02, '0.05': 0.03, '0.1': 0.05, '0.25': 0.09}
  llama3:8b      / base rate stated,
abstention allowed  : {'0.02': 0.0410958904109589, '0.05': 0.10975609756097561, '0.1': 0.05405405405405406, '0.25': 0.07352941176470588}
  granite3.3:8b  / base rate stated,
abstention allowed  : {'0.02': 0.030612244897959183, '0.05': 0.04, '0.1': 0.09, '0.25': 0.17}
  qwen3:8b       / base rate stated,
abstention allowed  : {'0.02': 0.0, '0.05': 0.0, '0.1': 0.025, '0.25': 0.02666666666666667}


**Step 2. The figure.** Drawn only from the values printed above, then written to `fig-prevalence-response.pdf`.

In [ ]:
# =============================================================================
# Fig 5 -- fig-prevalence-response
# =============================================================================
d = json.loads((ROOT / "accord/prevalence_replication/results/sweep_results.json").read_text())
models, prompts = d["models"], ["A_bare", "B_baserate_abstain"]
# The clinical comparison point is recomputed from the same two diagnostic
# files figures 1 and 2 draw, never typed.
_x01 = json.loads((ROOT / "results/accord/diag/step0_1_join_and_calibration.json").read_text())
_xtab = _x01["weak_confidence_is_degenerate"]["crosstab_label_x_confidence"]
clinical_y = _xtab["0.95"]["contradiction"] / (_xtab["0.95"]["contradiction"]
                                               + _xtab["1.0"]["not_contradiction"])
clinical_x = json.loads((ROOT / "results/accord/diag/step2_signal_check.json").read_text())[
    "contra_prob_vs_pharm_label__ambiguous_as_negative_n299"]["base_rate"]

PL = {"A_bare": "bare prompt", "B_baserate_abstain": "base rate stated,\nabstention allowed"}
# One hue per labeller, held down both rows of its column. granite3.3:8b takes
# AMBER: it used to draw in the green this set reserves for pharmacist truth,
# and a weak labeller wearing the truth colour is the kind of thing this thesis
# is about.
MODEL_HUE = {"llama3:8b": PLUM, "granite3.3:8b": AMBER, "qwen3:8b": COBALT}

fig, axes = plt.subplots(2, 3, figsize=(FIGW, 3.30), sharex=True, sharey=True)
canvas(fig)
fig.subplots_adjust(left=0.150, right=0.988, top=0.800, bottom=0.135,
                    wspace=0.14, hspace=0.20)
for r, pr in enumerate(prompts):
    for c, m in enumerate(models):
        ax = axes[r, c]
        xs, ys, lo, hi = [], [], [], []
        for p in ["0.02", "0.05", "0.1", "0.25"]:
            e = d["results"][m][pr].get(p)
            if not e:
                continue
            xs.append(float(p)); ys.append(e["positive_rate_of_decided"])
            # The sweep commits both counts behind every rate, so the interval
            # is a closed-form Wilson score interval on those counts.
            a, b = wilson_ci(e["n_contradiction"], e["n_decided"])
            lo.append(a); hi.append(b)
        hue = MODEL_HUE[m]
        # The wedge above the diagonal is over-prediction, and it is the whole
        # story of this grid, so it is drawn rather than left as white space.
        gx = np.linspace(0, 0.32, 2)
        ax.fill_between(gx, gx, 1.0, facecolor=MIST, edgecolor="none",
                        linewidth=0, zorder=0)
        ax.plot([0, 0.32], [0, 0.32], color=SLATE, linestyle=REF_DASH,
                linewidth=LW_REF, zorder=2)
        ax.errorbar(xs, ys, yerr=[[y - a for y, a in zip(ys, lo)],
                                  [b - y for y, b in zip(ys, hi)]],
                    fmt="-o", color=hue.CORE, ms=MS_DOT, lw=LW_DATA, zorder=4,
                    markeredgewidth=0, ecolor=hue.CORE,
                    elinewidth=LW_INTERVAL, capthick=LW_INTERVAL, capsize=3.2)
        if r == 0 and c == 0:
            case(ax.text(0.305, 0.965, "over-prediction", ha="right", va="top",
                         fontsize=T_NOTE, color=MUTE, zorder=5))
            # The clinical operating point: this grid's one scarlet mark, and
            # it appears once. Its label is a single line, which the ylim
            # change to 1.0 is what buys room for.
            ax.plot([clinical_x], [clinical_y], "s", color=SCARLET_CORE,
                    ms=MS_MARK, markeredgewidth=0, zorder=6)
            case(ax.text(0.29, clinical_y - 0.075,
                         f"clinical: {clinical_x * 100:.0f}% read as "
                         f"{clinical_y * 100:.0f}%",
                         ha="right", va="top", fontsize=T_SERIES,
                         color=SCARLET_DEEP, zorder=6))
        if r == 0 and c == 2:
            # The diagonal's angle on the page, not 45 degrees: the panel is
            # not square and the two axes do not share a range.
            bb = ax.get_window_extent()
            ang = math.degrees(math.atan2(0.32 / 1.0 * bb.height,
                                          0.32 / 0.32 * bb.width))
            # MUTE, not the SLATE of the line it names: SLATE is 3.48:1 on
            # white and this set does not set type in it.
            case(ax.text(0.175, 0.180, "perfect calibration", fontsize=T_NOTE,
                         color=MUTE, ha="center", va="bottom", rotation=ang,
                         rotation_mode="anchor", zorder=5))
        ax.set_xlim(0, 0.32); ax.set_ylim(0, 1.0)
        ax.set_xticks([0, 0.1, 0.2, 0.3]); ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
        if r == 0:
            ax.set_title(m, fontsize=T_SERIES, pad=5, color=MODEL_HUE[m].DEEP)
        if c == 0:
            ax.set_ylabel(PL[pr], fontsize=T_TICK, linespacing=1.3)
        ax.tick_params(labelsize=T_TICK)
        frame(ax)
fig.supxlabel("True prevalence", fontsize=T_AXIS, y=0.018)
fig.supylabel("Labeller predicted-positive rate", fontsize=T_AXIS, x=0.006)
signature(fig, SCARLET_DEEP,
          "No labeller tracks prevalence. Stating the base rate does not fix it.")
fig.savefig(HERE / "fig-prevalence-response.pdf"); plt.close(fig)
print("wrote fig-prevalence-response.pdf")


## Figure 6: `fig-falsification`


In [15]:
p_agg = os.path.join(RESULTS_ACCORD, "decontam", "accord_v2_decontam_gold_aggregate.json")
d_agg = json.load(open(p_agg))
aN, aD = d_agg["aggregate"]["alpha_n"]["mean"], d_agg["aggregate"]["alpha_d"]["mean"]
sN, sD = d_agg["aggregate"]["alpha_n"]["std"], d_agg["aggregate"]["alpha_d"]["std"]

p_transfer = os.path.join(RESULTS_ACCORD, "generality", "transfer_results_correct.json")
d_transfer = json.load(open(p_transfer))
f1 = [d_transfer["hf_reference"]["rule_type_f1"], d_transfer["sepsis"]["rule_type_f1"], d_transfer["aki"]["rule_type_f1"]]
cohort_labels = ["HF\n(in domain)", "sepsis", "AKI"]
n_cand = [d_transfer["hf_reference"]["n_candidates"], d_transfer["sepsis"]["n_candidates"],
          d_transfer["aki"]["n_candidates"]]
# The three seeds behind each mean, read from the same file the means come from.
seeds = sorted(d_agg["per_seed"])
seed_N = [d_agg["per_seed"][s]["authority_scale"]["alpha_nursing"] for s in seeds]
seed_D = [d_agg["per_seed"][s]["authority_scale"]["alpha_discharge"] for s in seeds]

p_audit = os.path.join(RESULTS_ACCORD, "number_provenance_audit.json")
d_audit = json.load(open(p_audit))
claim = next(r for r in d_audit["records"] if "45.2" in r["claim"])
# The 45.2% held-out rule-accuracy reference line has been REMOVED from this
# figure: 0.4515 appears in no result file anywhere on disk, only in figure code
# and in the audit record printed below, which flags it as untraceable.

print("fig-falsification: values plotted")
print(f"  alpha_N (nursing authority scale), 3-seed mean +/- std : {aN:.4f} +/- {sN:.4f}")
print(f"  alpha_D (discharge authority scale), 3-seed mean +/- std: {aD:.4f} +/- {sD:.4f}")
for lab, v, nc in zip(cohort_labels, f1, n_cand):
    print(f"  rule-type F1, {lab.replace(chr(10), ' '):16s}: {v:.4f}  (n={nc})")
print(f"  alpha_N per seed: {['%.4f' % v for v in seed_N]}")
print(f"  alpha_D per seed: {['%.4f' % v for v in seed_D]}")
print("  alpha initialisation 0.5 / 1.0: a stated design constant, not a measurement;")
print("    it is in no result file, and the figure legend now says so.")
print("  NOT plotted (removed): held-out rule-prediction accuracy 0.4515")
print(f"  number_provenance_audit.json claim record: {claim}")

fig-falsification: values plotted
  alpha_N (nursing authority scale), 3-seed mean +/- std : 0.5076 +/- 0.0006
  alpha_D (discharge authority scale), 3-seed mean +/- std: 1.0038 +/- 0.0010
  rule-type F1, HF (in domain)  : 0.9603  (n=1978)
  rule-type F1, sepsis          : 0.0654  (n=775)
  rule-type F1, AKI             : 0.0595  (n=1983)
  alpha_N per seed: ['0.5070', '0.5074', '0.5084']
  alpha_D per seed: ['1.0024', '1.0042', '1.0047']
  alpha initialisation 0.5 / 1.0: a stated design constant, not a measurement;
    it is in no result file, and the figure legend now says so.
  NOT plotted (removed): held-out rule-prediction accuracy 0.4515
  number_provenance_audit.json claim record: {'claim': '45.2% held-out rule-prediction accuracy (recomputes to 0.4515)', 'tex_location': 'ch6:291', 'artifact': 'gold-standard CSV per ch6 caption', 'artifact_exists': False, 'reported': 0.452, 'recomputed': None, 'verdict': 'UNTRACEABLE', 'blast_radius': 'body', 'note': "Caption asserts it 'recompu

**Step 2. The figure.** Drawn only from the values printed above, then written to `fig-falsification.pdf`.

In [ ]:
# ---- fig:falsification -----------------------------------------------------
agg_all = json.loads((SUPP / "accord_v2_decontam_gold_aggregate.json").read_text())
agg = agg_all["aggregate"]
aN, aD = agg["alpha_n"]["mean"], agg["alpha_d"]["mean"]
seeds = sorted(agg_all["per_seed"])
seed_N = [agg_all["per_seed"][s_]["authority_scale"]["alpha_nursing"] for s_ in seeds]
seed_D = [agg_all["per_seed"][s_]["authority_scale"]["alpha_discharge"] for s_ in seeds]
INIT_N, INIT_D = 0.5, 1.0          # design constants, CORRECTIONS.md C1

transfer = json.loads((ROOT / "results/accord/generality/transfer_results_correct.json").read_text())
f1 = [transfer["hf_reference"]["rule_type_f1"], transfer["sepsis"]["rule_type_f1"],
      transfer["aki"]["rule_type_f1"]]
n_cand = [transfer["hf_reference"]["n_candidates"], transfer["sepsis"]["n_candidates"],
          transfer["aki"]["n_candidates"]]

fig, (a, b) = plt.subplots(1, 2, figsize=(FIGW, 2.45),
                           gridspec_kw={"width_ratios": [1.06, 1.0]})
canvas(fig)
fig.subplots_adjust(left=0.128, right=0.985, top=0.700, bottom=0.235, wspace=0.34)

# ---- (a) the dumbbell ------------------------------------------------------
TOL = 0.02
a.axvspan(1 - TOL, 1 + TOL, facecolor=SCARLET_TINT, edgecolor="none",
          linewidth=0, zorder=0)
moved = []
for i, (init, mean, sds) in enumerate(((INIT_N, aN, seed_N),
                                       (INIT_D, aD, seed_D))):
    ratio = mean / init
    moved.append(abs(ratio - 1))
    a.plot([1.0, ratio], [i, i], color=GRAPHITE, lw=2.4,
           solid_capstyle="butt", zorder=3)
    # The three seeds spread 10.8pt about a 6.6pt marker, so on the dumbbell's
    # own line they were neither hidden nor visible: a grey crescent behind the
    # final dot that reads as a print smudge. They drop to their own sub-row,
    # true x, where three dots read as three seeds.
    a.plot([s_ / init for s_ in sds], [i - 0.32] * len(sds), "o", color=SLATE,
           ms=MS_SEED, markeredgewidth=0, zorder=5)
    a.plot([1.0], [i], "o", markerfacecolor=PAPER, markeredgecolor=GRAPHITE,
           markeredgewidth=1.2, ms=MS_DOT, zorder=4)
    a.plot([ratio], [i], "o", color=GRAPHITE, ms=MS_DOT, zorder=6)
    a.text(ratio + 0.0026, i, f"{(ratio - 1) * 100:+.1f}%",
           fontsize=T_NOTE, color=MUTE, va="center", ha="left", zorder=6)
case(a.text(1 - TOL + 0.0012, 2.00,
            f"reported as learned:\nneither left a {TOL * 100:.0f}% band",
            fontsize=T_NOTE, color=SCARLET_DEEP, ha="left", va="top",
            linespacing=1.24, zorder=6))
a.set_yticks([1, 0])
a.set_yticklabels([r"$\alpha_D$" + "\n(set to 1.00)",
                   r"$\alpha_N$" + "\n(set to 0.50)"],
                  fontsize=T_SERIES, linespacing=1.25)
a.set_ylim(-0.72, 2.10)
a.set_xlim(0.9755, 1.0295)
a.set_xticks([0.98, 0.99, 1.00, 1.01, 1.02])
a.set_xticklabels(["0.98", "0.99", "1.00", "1.01", "1.02"])
a.set_xlabel("Final value, as a multiple\nof the initialisation",
             linespacing=1.25)
a.tick_params(axis="y", length=0)
frame(a)

# ---- (b) the cliff ---------------------------------------------------------
# Cohorts are not entities, so all three columns take one hue. The two the head
# was never fitted on are BARE: no interval for either is committed anywhere,
# because the per-pair scores they would be resampled from are not released.
xs = [0, 1, 2]
fills = [COBALT_TINT, COBALT_BARE, COBALT_BARE]
b.axvspan(0.5, 2.6, facecolor=SCARLET_WASH, edgecolor="none", linewidth=0,
          zorder=0)
b.bar(xs, f1, width=0.62, color=fills, edgecolor=COBALT_DEEP,
      linewidth=LW_BAR, zorder=3)
case(b.text(0.58, 0.985, "not fitted\nhere", ha="left", va="top",
            linespacing=1.24,
            fontsize=T_NOTE, color=SCARLET_DEEP, zorder=5))
# The drop itself, from the top of the column the head was fitted on to the
# top of the first column it was not.
b.plot([0, 1], [f1[0], f1[1]], color=SCARLET_CORE, lw=LW_DATA, zorder=5)
# The drop, written as a claim numeral, and written vertically, because that
# is the shape of the finding. The arrow is a patch: never a dash, never two
# keyboard characters.
b.text(1.88, 0.86, f"{f1[0]:.2f}", ha="center", va="center", fontsize=T_CLAIM,
       fontweight="bold", color=SCARLET_DEEP, zorder=6)
arrow(b, (1.88, 0.73), (1.88, 0.48), color=SCARLET_CORE, lw=LW_DATA, zorder=6)
b.text(1.88, 0.35, f"{f1[1]:.2f}", ha="center", va="center", fontsize=T_CLAIM,
       fontweight="bold", color=SCARLET_DEEP, zorder=6)
b.set_xticks(xs)
b.set_xticklabels([f"HF,\nin domain\nn = {n_cand[0]:,}",
                   f"sepsis\nn = {n_cand[1]:,}",
                   f"AKI\nn = {n_cand[2]:,}"], fontsize=T_NOTE,
                  linespacing=1.28)
b.set_xlim(-0.62, 2.6)
b.set_ylim(0, 1.02)
b.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
b.set_ylabel(r"Rule-type $F_1$")
frame(b)

for ax_, tag in ((a, "(a)"), (b, "(b)")):
    panel_label(fig, ax_, tag)
signature(fig, SCARLET_DEEP,
          "Two parameters reported as learned did not move;\n"
          "the rule head does not transfer.")
fig.savefig(HERE / "fig-falsification.pdf"); plt.close(fig)
man["fig:falsification"] = {"file": "fig-falsification.pdf",
    "source": "accord_supp/results/accord_v2_decontam_gold_aggregate.json (a); "
              "results/accord/generality/transfer_results_correct.json (b)",
    "values": {"alpha_n": aN, "alpha_d": aD, "alpha_n_per_seed": seed_N,
               "alpha_d_per_seed": seed_D, "rule_type_f1": f1,
               "n_candidates": n_cand,
               "alpha_init_stated_design_constant": [INIT_N, INIT_D],
               "alpha_n_ratio": aN / INIT_N, "alpha_d_ratio": aD / INIT_D},
    "removed": "45.2% held-out rule-accuracy reference line (0.4515 is in no result file)"}


## Figure 7: `fig-power`


In [17]:
p_power = os.path.join(RESULTS_ACCORD, "section6_auroc_power.json")
pw = json.load(open(p_power))
mda = pw["minimum_detectable_auc"]
n_pos_study = pw["n_positives"]
n_neg_study = pw["n_negatives"]
alpha_study = pw["alpha"]
power_study = pw["power"]

p_task_runs = os.path.join(RESULTS, "task_runs.json")
d_task_runs = json.load(open(p_task_runs))
mort_lineage = d_task_runs["concat_s1"]["data_lineage"]
mort_n_test = mort_lineage["test_n"]
mort_n_pos = mort_lineage["test_n_pos"]
mort_n_neg = mort_n_test - mort_n_pos

def hanley_mde(n_pos, n_neg, alpha=alpha_study, power=power_study):
    """Smallest AUROC distinguishable from 0.5 at the given power, Hanley-McNeil variance,
    using both class counts."""
    z = norm.ppf(1 - alpha / 2) + norm.ppf(power)
    out = []
    for m, n in zip(np.atleast_1d(n_pos), np.atleast_1d(n_neg)):
        lo, hi = 0.5, 0.9999
        for _ in range(80):
            a = (lo + hi) / 2
            q1, q2 = a / (2 - a), 2 * a * a / (1 + a)
            v = (a * (1 - a) + (m - 1) * (q1 - a * a) + (n - 1) * (q2 - a * a)) / (m * n)
            if (a - 0.5) / math.sqrt(max(v, 1e-12)) < z:
                lo = a
            else:
                hi = a
        out.append((lo + hi) / 2)
    return np.array(out)

print("fig-power: values plotted")
print(f"  this study: n_positives={n_pos_study}, n_negatives={n_neg_study}, alpha={alpha_study}, power={power_study}")
mde_study = hanley_mde([n_pos_study], [n_neg_study])[0]
print(f"  this study minimum detectable AUROC, stored in the file: {mda:.4f}")
print(f"  this study minimum detectable AUROC, from the plotted curve: {mde_study:.4f}  <- the marker uses this")
print("    (the stored value is not reproduced by this curve; the marker is placed on the curve")
print("     the axes actually draw, so the point and the line agree)")
print(f"  mortality task (task_runs.json / concat_s1 / data_lineage): test_n={mort_n_test}, test_n_pos={mort_n_pos}, test_n_neg={mort_n_neg}")
mort_mde = hanley_mde([mort_n_pos], [mort_n_neg])[0]
print(f"  mortality task minimum detectable AUROC: {mort_mde:.4f}")

n_pos_grid = np.logspace(np.log10(5), np.log10(3000), 60)
neg_pos_ratio = n_neg_study / n_pos_study   # this evaluation set's own positive:negative ratio
n_neg_grid = n_pos_grid * neg_pos_ratio
mde_grid = hanley_mde(n_pos_grid, n_neg_grid)

fig-power: values plotted
  this study: n_positives=12, n_negatives=241, alpha=0.05, power=0.8
  this study minimum detectable AUROC, stored in the file: 0.7382
  this study minimum detectable AUROC, from the plotted curve: 0.7355  <- the marker uses this
    (the stored value is not reproduced by this curve; the marker is placed on the curve
     the axes actually draw, so the point and the line agree)
  mortality task (task_runs.json / concat_s1 / data_lineage): test_n=6776, test_n_pos=570, test_n_neg=6206
  mortality task minimum detectable AUROC: 0.5360


**Step 2. The figure.** Drawn only from the values printed above, then written to `fig-power.pdf`.

In [ ]:
# =============================================================================
# Fig 7 -- fig-power
# =============================================================================
pw = json.loads((ROOT / "results/accord/section6_auroc_power.json").read_text())
n_pos_study, n_neg_study = pw["n_positives"], pw["n_negatives"]
alpha_study, power_study = pw["alpha"], pw["power"]
_mort = json.loads((ROOT / "results/task_runs.json").read_text())["concat_s1"]["data_lineage"]
mort_n_pos = _mort["test_n_pos"]
mort_n_neg = _mort["test_n"] - mort_n_pos


# The significance level is named alpha_level, not alpha: this set bans the
# matplotlib alpha= keyword outright and verification greps for it, so a
# statistical alpha must not wear the same name as a transparency.
def hanley_mde(n_pos, n_neg, alpha_level=0.05, power=0.80):
    """Smallest AUROC distinguishable from 0.5 at the given power, Hanley and
    McNeil variance. Uses BOTH class counts."""
    from scipy.stats import norm
    z = norm.ppf(1 - alpha_level / 2) + norm.ppf(power)
    out = []
    for m_, n_ in zip(np.atleast_1d(n_pos), np.atleast_1d(n_neg)):
        lo, hi = 0.5, 0.9999
        for _ in range(80):
            a = (lo + hi) / 2
            q1, q2 = a / (2 - a), 2 * a * a / (1 + a)
            v = (a * (1 - a) + (m_ - 1) * (q1 - a * a) + (n_ - 1) * (q2 - a * a)) / (m_ * n_)
            if (a - 0.5) / math.sqrt(max(v, 1e-12)) < z:
                lo = a
            else:
                hi = a
        out.append((lo + hi) / 2)
    return np.array(out)


n_pos = np.logspace(np.log10(5), np.log10(3000), 60)
n_neg = n_pos * (n_neg_study / n_pos_study)
mde = hanley_mde(n_pos, n_neg, alpha_level=alpha_study, power=power_study)
mde_study = hanley_mde([n_pos_study], [n_neg_study], alpha_level=alpha_study,
                       power=power_study)[0]
mde_mort = hanley_mde([mort_n_pos], [mort_n_neg], alpha_level=alpha_study,
                      power=power_study)[0]

fig, ax = plt.subplots(figsize=(FIGW, 2.75))
canvas(fig)
fig.subplots_adjust(left=0.128, right=0.988, top=0.795, bottom=0.180)
curve = np.clip(mde, 0.5, 1.0)
# Two grounds, hard boundary at the curve. The region above the curve is what
# this evaluation could have resolved; the region under it is what it could
# not, which is where this thesis's own evaluation sits.
ax.fill_between(n_pos, curve, 1.0, facecolor=EMERALD_WASH, edgecolor="none",
                linewidth=0, zorder=1)
ax.fill_between(n_pos, 0.5, curve, facecolor=SCARLET_WASH, edgecolor="none",
                linewidth=0, zorder=1)
# The MDE curve is measured, but it is not an entity, so it takes the
# structural measurement tone.
ax.plot(n_pos, curve, color=GRAPHITE, lw=LW_DATA, zorder=4)
ax.set_xscale("log")
ax.set_xlim(5, 3000)
ax.set_ylim(0.5, 1.0)
ax.set_yticks([0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
ax.set_xlabel("Positive examples in the evaluation set (log scale)")
ax.set_ylabel(f"Minimum detectable AUROC\n({power_study * 100:.0f}% power)",
              linespacing=1.3)
case(ax.text(2700, 0.965, "detectable", fontsize=T_SERIES, color=EMERALD_DEEP,
             ha="right", va="top", zorder=5))
case(ax.text(6.0, 0.515, "not detectable at this sample size",
             fontsize=T_SERIES, color=SCARLET_DEEP, ha="left", va="bottom",
             zorder=5))
# Both marks are labelled adjacent to themselves. No leaders anywhere.
ax.plot([mort_n_pos], [mde_mort], "s", color=SLATE, ms=MS_MARK,
        markeredgewidth=0, zorder=6)
case(ax.text(mort_n_pos, mde_mort + 0.022,
             f"mortality control, {mort_n_pos} positives", fontsize=T_SERIES,
             color=MUTE, ha="center", va="bottom", zorder=6))
ax.plot([n_pos_study], [mde_study], "o", color=SCARLET_CORE, ms=MS_DOT + 1.4,
        markeredgewidth=0, zorder=6)
case(ax.text(n_pos_study * 1.5, mde_study + 0.030,
             f"this study, {n_pos_study} positives", fontsize=T_SERIES,
             color=SCARLET_DEEP, ha="left", va="bottom", zorder=6))
frame(ax)
signature(fig, SCARLET_DEEP,
          f"At {n_pos_study} positives, nothing below AUROC {mde_study:.3f} "
          f"was ever detectable.")
fig.savefig(HERE / "fig-power.pdf"); plt.close(fig)
print("wrote fig-power.pdf")


## Figure 8: `fig-forest`


In [ ]:
iv = json.load(open(os.path.join(
    RESULTS_ACCORD, "stats", "intervals.json")))["E2_mcc_intervals"]
dec = json.load(open(os.path.join(
    RESULTS_ACCORD, "decontam", "corrected_gold_metrics.json")))["aggregate"]
df34 = json.load(open(os.path.join(
    RESULTS_ACCORD, "control", "final_report.json")))[
        "ensemble_mean_of_seed_scores"]["gold_defect_free_rule34"]
tr = json.load(open(os.path.join(RESULTS, "task_runs.json")))
sc = json.load(open(os.path.join(
    RESULTS_ACCORD, "scaling", "scaling_results.json")))

print("fig-forest: every headline estimate in the thesis, read against zero")
for k, v in iv.items():
    print(f"  MCC {k:32s} {v['mcc']:+.4f}  [{v['ci95'][0]:+.4f}, {v['ci95'][1]:+.4f}]")
print(f"  AUROC decontaminated ensemble        {dec['ensemble_auroc']-0.5:+.4f}"
      f"  [{dec['ensemble_ci95'][0]-0.5:+.4f}, {dec['ensemble_ci95'][1]-0.5:+.4f}]")
print(f"  AUROC defect-free stratum            {df34['auroc']-0.5:+.4f}"
      f"  [{df34['bootstrap']['ci95'][0]-0.5:+.4f}, {df34['bootstrap']['ci95'][1]-0.5:+.4f}]")
print("  pretraining gaps are recomputed in the figure cell from task_runs.json"
      " and scaling_results.json")


**Step 2. The figure.** Drawn only from the values printed above, then written to `fig-forest.pdf`.

In [ ]:
# =============================================================================
# Fig 8 -- fig-forest. The master summary: every headline estimate in the
# thesis, and the only place they appear.
# =============================================================================
iv = json.loads((ROOT / "results/accord/stats/intervals.json").read_text())["E2_mcc_intervals"]
NICE = {"gpt4o_zeroshot": "GPT-4o zero-shot", "gpt4o_fewshot": "GPT-4o few-shot",
        "accord_binary_at_0.5_253core": "ACCORD at 0.5",
        "llama_baseline": "Llama 3 8B"}
# Rows take their ENTITY hue, which they did not before: family 1's four rows
# all drew one blue, contradicting the graphical abstract 85 pages earlier.
ROW_HUE = {"gpt4o_zeroshot": COBALT, "gpt4o_fewshot": AZURE,
           "accord_binary_at_0.5_253core": AMBER, "llama_baseline": PLUM}
fam_mcc = [((NICE.get(k, k) + f" ({v['n']} pairs)") if k.startswith("accord")
            else NICE.get(k, k), v["mcc"], v["ci95"][0], v["ci95"][1],
            ROW_HUE[k]) for k, v in iv.items()]
_dec = json.loads((ROOT / "results/accord/decontam/corrected_gold_metrics.json").read_text())["aggregate"]
_ctrl = json.loads((ROOT / "results/accord/control/final_report.json").read_text())
_df34 = _ctrl["ensemble_mean_of_seed_scores"]["gold_defect_free_rule34"]
fam_auroc = [
    ("Decontaminated ensemble", _dec["ensemble_auroc"] - 0.5,
     _dec["ensemble_ci95"][0] - 0.5, _dec["ensemble_ci95"][1] - 0.5, AMBER),
    ("Defect-free stratum", _df34["auroc"] - 0.5,
     _df34["bootstrap"]["ci95"][0] - 0.5, _df34["bootstrap"]["ci95"][1] - 0.5,
     EMERALD),
]
_pb2 = [json.loads((ROOT / f"local_only/task_runs/concat_s{s}.json").read_text())["auroc"] for s in (1, 2, 3)]
_ou2 = [json.loads(pathlib.Path(f).read_text())["auroc"]
        for f in sorted(glob.glob(str(ROOT / "local_only/task_runs/ours_current_s*.json")))]
_g = st.mean(_pb2) - st.mean(_ou2)
_se = math.sqrt(st.stdev(_pb2) ** 2 / 3 + st.stdev(_ou2) ** 2 / 3)
_sc = json.loads((ROOT / "results/accord/scaling/scaling_results.json").read_text())
_p100 = [x["accuracy"] for x in _sc if x["tag"] == "scratch_100M"]
_pbert = [x["accuracy"] for x in _sc if x["tag"] == "pretrained_bert"]
_g44 = st.mean(_pbert) - st.mean(_p100)
_se44 = math.sqrt(st.stdev(_pbert) ** 2 / 3 + st.stdev(_p100) ** 2 / 3)
fam_gap = [
    ("In-hospital mortality, AUROC", _g, _g - 1.96 * _se, _g + 1.96 * _se, EMERALD),
    ("AG News at 100M, accuracy", _g44, _g44 - 1.96 * _se44, _g44 + 1.96 * _se44,
     EMERALD),
]
# A family is not an entity, so the family band is a structural ground and the
# family heading is set in the annotation grey, not in a fake hue.
FAMILIES = [("Matthews correlation", fam_mcc, MIST),
            ("AUROC above chance", fam_auroc, EMERALD_WASH),
            ("Pretraining gap, pretrained minus from scratch", fam_gap, AMBER_WASH)]
for _, rs, _b in FAMILIES:
    rs.sort(key=lambda r: r[1])

placed, bands_, heads, y = [], [], [], 0.0
for fi, (fam, rs, bg) in enumerate(reversed(FAMILIES)):
    y0 = y - 0.55
    for r in rs:
        placed.append((y, r)); y += 1.0
    y -= 1.0
    heads.append((y + 0.52, fam))
    bands_.append((y0, y + 1.05, bg))
    y += 1.55
rows = [r for _, r in placed]
y_ceiling = placed[-1][0] + 1.10

fig, ax = plt.subplots(figsize=(FIGW, 3.40))
canvas(fig)
fig.subplots_adjust(left=0.325, right=0.988, top=0.855, bottom=0.128)
X_LO, X_HI = -0.40, 1.06
ax.set_xlim(X_LO, X_HI)
ax.set_ylim(-0.80, y_ceiling)
ax.set_xticks([-0.3, -0.2, -0.1, 0.0, 0.1, 0.2, 0.3, 0.4])
frame(ax, grid="x")
for y0, y1, bg in bands_:
    ax.axhspan(y0, y1, facecolor=bg, edgecolor="none", linewidth=0, zorder=0)
# The zero line is the spine of this figure, not a reference to be looked past.
ax.axvline(0, color=GRAPHITE, lw=1.2, zorder=2)

# The drawn half-width of an interval, in points, decides whether the sliver
# rule fires. Anything under 1.5pt is invisible beside its own marker, and an
# invisible interval with no cue is the failure this set's first rule exists to
# prevent.
plot_w_pt = (0.988 - 0.325) * FIGW * 72
sliver = []
for yy, (lab, e, lo, hi, hue) in placed:
    half_pt = (hi - lo) / 2 / (X_HI - X_LO) * plot_w_pt
    ax.errorbar([e], [yy], xerr=[[e - lo], [hi - e]], fmt="o", color=hue.CORE,
                zorder=5, **ERRBAR, **DOT)
    mark = ""
    if half_pt < 1.5:
        for b in (lo, hi):
            ax.plot([b, b], [yy - 0.19, yy + 0.19], color=hue.DEEP, lw=1.5,
                    solid_capstyle="butt", zorder=6)
        mark = "‡ "
        sliver.append(lab)
    ax.text(X_HI - 0.02, yy, f"{mark}{e:+.4f}  [{lo:+.3f}, {hi:+.3f}]",
            ha="right", va="center", fontsize=T_SERIES, color=hue.DEEP,
            zorder=6)

gt = ax.get_yaxis_transform()
ax.set_yticks([yy for yy, _ in placed])
ax.set_yticklabels([r[0] for r in rows], fontsize=T_SERIES, linespacing=1.25)
for yh, fam in heads:
    ax.text(-0.385, yh, fam, ha="left", va="bottom", fontsize=T_SERIES,
            color=MUTE, style="italic", zorder=4)
ax.set_xlabel("Effect, 95% interval (unit named per family)")
ax.tick_params(axis="y", length=0)
if sliver:
    ax.text(-0.385, -0.44, "‡ interval narrower than the marker",
            ha="left", va="center", fontsize=T_NOTE, color=MUTE, zorder=6)
n_cross = sum(1 for r in rows if r[2] <= 0 <= r[3])
signature(fig, GRAPHITE,
          f"{n_cross} of the {len(rows)} headline estimates in this thesis "
          f"cross zero.")
fig.savefig(HERE / "fig-forest.pdf"); plt.close(fig)
print(f"wrote fig-forest.pdf ({n_cross}/{len(rows)} cross zero; "
      f"sliver rows: {sliver})")


## Figure 9: `fig-scaling`


In [25]:
p_scaling = os.path.join(RESULTS_ACCORD, "scaling", "scaling_results.json")
d_scaling = json.load(open(p_scaling))
by, params_by_tag = {}, {}
for x in d_scaling:
    by.setdefault(x["tag"], []).append(x["accuracy"])
    params_by_tag[x["tag"]] = x["non_emb_params"]
tags = ["scratch_10M", "scratch_30M", "scratch_100M"]
xs = [params_by_tag[t] for t in tags]
mu = [statistics.mean(by[t]) for t in tags]
sd = [statistics.stdev(by[t]) for t in tags]
pm, ps, pp = statistics.mean(by["pretrained_bert"]), statistics.stdev(by["pretrained_bert"]), params_by_tag["pretrained_bert"]

p_task_runs = os.path.join(RESULTS, "task_runs.json")
d_task_runs = json.load(open(p_task_runs))
pb = [d_task_runs[f"concat_s{s}"]["auroc"] for s in (1, 2, 3)]
ou = [d_task_runs[f"ours_current_s{s}"]["auroc"] for s in (0, 1, 2)]
pbm, pbs = statistics.mean(pb), statistics.stdev(pb)
oum, ous = statistics.mean(ou), statistics.stdev(ou)
gap = pbm - oum
clinical_params = d_task_runs["ours_current_s0"]["non_emb_params"]

print("fig-scaling: values plotted")
print(f"  AG News: pretrained_bert mean +/- std = {pm:.4f} +/- {ps:.4f}  ({pp / 1e6:.0f}M params)")
for t, x, m, s in zip(tags, xs, mu, sd):
    print(f"  AG News: {t:14s} mean +/- std = {m:.4f} +/- {s:.4f}  ({x / 1e6:.1f}M params)  gap to pretrained = {pm - m:+.4f}")
print(f"  clinical task: PubMedBERT (concat_s1..3) AUROC mean +/- std = {pbm:.4f} +/- {pbs:.4f}")
print(f"  clinical task: from-scratch (ours_current_s0..2) AUROC mean +/- std = {oum:.4f} +/- {ous:.4f}  ({clinical_params / 1e6:.1f}M params)")
print(f"  clinical task gap: {gap:+.4f}")

fig-scaling: values plotted
  AG News: pretrained_bert mean +/- std = 0.9396 +/- 0.0010  (86M params)
  AG News: scratch_10M    mean +/- std = 0.8790 +/- 0.0040  (10.6M params)  gap to pretrained = +0.0606
  AG News: scratch_30M    mean +/- std = 0.8833 +/- 0.0009  (31.9M params)  gap to pretrained = +0.0563
  AG News: scratch_100M   mean +/- std = 0.8878 +/- 0.0016  (99.2M params)  gap to pretrained = +0.0518
  clinical task: PubMedBERT (concat_s1..3) AUROC mean +/- std = 0.7693 +/- 0.0102
  clinical task: from-scratch (ours_current_s0..2) AUROC mean +/- std = 0.7662 +/- 0.0073  (10.6M params)
  clinical task gap: +0.0031


**Step 2. The figure.** Drawn only from the values printed above, then written to `fig-scaling.pdf`.

In [ ]:
# =============================================================================
# Fig 9 -- fig-scaling. Panel (b) deleted: its content is Figure 8's last row.
# =============================================================================
sc = json.loads((ROOT / "results/accord/scaling/scaling_results.json").read_text())
by, pr_ = {}, {}
for x in sc:
    by.setdefault(x["tag"], []).append(x["accuracy"]); pr_[x["tag"]] = x["non_emb_params"]
tags = ["scratch_10M", "scratch_30M", "scratch_100M"]
xs = [pr_[t] for t in tags]; mu = [st.mean(by[t]) for t in tags]
sd = [st.stdev(by[t]) for t in tags]
pm, ps, pp = st.mean(by["pretrained_bert"]), st.stdev(by["pretrained_bert"]), pr_["pretrained_bert"]

fig, ax = plt.subplots(figsize=(FIGW, 2.60))
canvas(fig)
fig.subplots_adjust(left=0.132, right=0.988, top=0.755, bottom=0.180)
# The pretrained arm is a measured arm with three seeds and a spread of its
# own, not a chance line, so its band, its centre line and its label are all
# one hue and that hue is the reference-standard green.
ax.axhspan(pm - ps, pm + ps, facecolor=EMERALD_WASH, edgecolor="none",
           linewidth=0, zorder=1)
ax.axhline(pm, color=EMERALD_CORE, lw=LW_DATA, zorder=2)
# The gap between the two arms IS this figure's subject, and it was white
# space. It is the region between two measured series, so it is drawn as one.
ax.fill_between(xs, mu, [pm] * len(xs), facecolor=MIST, edgecolor="none",
                linewidth=0, zorder=1)
ax.errorbar(xs, mu, yerr=sd, fmt="none", ecolor=COBALT_CORE, zorder=3, **ERRBAR)
# SEED DOTS ARE DRAWN ON THE FROM-SCRATCH ARM ONLY, and the pretrained arm is
# the reason. Size is the variable on the scratch arm, so each of its points has
# an x and its three seeds stack on it. The pretrained arm is not a point on
# that sweep: it is a reference level, which is why it is drawn as a band across
# the WHOLE width rather than a mark at 8.6e7. Its three seeds are already on
# the canvas twice over -- they are what the band's half-height and the label's
# "plus or minus 0.0010" are computed from. Drawn a third time as dots they
# landed on the band's own centre line: two vanished under the 1.8pt stroke and
# the third cleared it by 2pt, leaving one isolated slate fleck grazing the
# label. Z-order cannot fix that (the third seed lies outside mean +/- sd, so it
# clears the band as well as the line) and jitter would put a false parameter
# count under a log axis. The band is the honest drawing of these three numbers.
for x, t in zip(xs, tags):
    ax.plot([x] * len(by[t]), by[t], "o", color=SLATE, ms=MS_SEED,
            markeredgewidth=0, zorder=4)
ax.plot(xs, mu, "-o", color=COBALT_CORE, ms=MS_DOT, lw=LW_DATA, zorder=5)
for x, m_, s_ in zip(xs, mu, sd):
    ax.annotate(f"{pm - m_:+.4f}", (x, m_ - s_), textcoords="offset points",
                xytext=(0, -11), ha="center", va="top", fontsize=T_NOTE,
                color=MUTE)
ax.set_xscale("log")
ax.set_xlabel("Non-embedding parameters (log scale)")
ax.set_ylabel("Accuracy, AG News")
ax.set_ylim(0.851, 0.950)
ax.set_yticks([0.86, 0.88, 0.90, 0.92, 0.94])
# The xlim runs 8% further right than the pretrained point, which is the room
# the label needs to sit inside the box at the line's own right end.
ax.set_xlim(7.0e6, 2.1e8)
case(ax.text(2.0e8, pm - 0.0034,
             f"pretrained BERT {pm:.4f} plus or minus {ps:.4f}",
             fontsize=T_SERIES, color=EMERALD_DEEP, va="top", ha="right",
             zorder=6))
# The subject of the figure, named inside itself. MUTE on MIST is 4.9:1.
ax.text(5.0e7, 0.9150, "the gap", fontsize=T_NOTE, color=MUTE,
        ha="center", va="center", zorder=6)
frame(ax, grid=True)
ax.plot(xs, mu, "-o", color=COBALT_CORE, ms=MS_DOT, lw=LW_DATA, zorder=5)
case(ax.text(xs[1], mu[1] + 0.0060, "from scratch", fontsize=T_SERIES,
             color=COBALT_DEEP, va="bottom", ha="center", zorder=6))
signature(fig, EMERALD_DEEP,
          "The from-scratch arm rises with size\n"
          "and never reaches the pretrained band.")
fig.savefig(HERE / "fig-scaling.pdf"); plt.close(fig)
print("wrote fig-scaling.pdf")


## Figure 10: `fig-threshold-surface`


In [27]:
p_thresh = os.path.join(RESULTS_ACCORD, "selection", "threshold_robustness_results.json")
d_thresh = json.load(open(p_thresh))
df = pd.DataFrame(d_thresh["cells"])
piv = df.pivot_table(index="achieved_rate", columns="beta", values="demographic_parity_gap", aggfunc="mean")
mde = df.pivot_table(index="achieved_rate", columns="beta", values="mde_demographic_parity_gap", aggfunc="mean")
R, B, Z, M = piv.index.values, piv.columns.values, piv.values, mde.values
e9_rate = d_thresh["original_e9_population_pred_pos_rate"]

# Named setting, not read from this file: the audit's own demographic-parity tolerance.
TOLERANCE = 0.05

print("fig-threshold-surface: values plotted")
print(f"  grid: {Z.shape[0]} operating points x {Z.shape[1]} selection strengths = {Z.size} cells")
print(f"  demographic-parity gap range: [{np.nanmin(Z):.4f}, {np.nanmax(Z):.4f}]")
print(f"  minimum-detectable-gap range: [{np.nanmin(M):.4f}, {np.nanmax(M):.4f}]")
n_under = int((M > TOLERANCE).sum())
print(f"  cells underpowered at TOLERANCE={TOLERANCE}: {n_under} of {M.size}")
print(f"  original E9 operating point (predicted-positive rate): {e9_rate:.4f}")

# Viridis: perceptually uniform, monotone in lightness end to end, and safe
# under every common dichromacy, which is why it is the standard for a field
# like this one. It replaces a five-stop ramp in a single blue. The ramp was
# defensible and it was also the reason a surface of 270 measured cells read as
# one wash: a single hue can only spend lightness, so the interesting middle of
# this field, between about 0.04 and 0.12, occupied one narrow band of pale
# blue. Viridis spends hue as well, so the same middle is a visible traverse
# from blue through teal to green, and the greyscale reading survives because
# viridis is still monotone in lightness: dark is low, light is high, in print
# and on paper both.
cmap = plt.get_cmap("viridis")

fig-threshold-surface: values plotted
  grid: 18 operating points x 15 selection strengths = 270 cells
  demographic-parity gap range: [0.0021, 0.2001]
  minimum-detectable-gap range: [0.0305, 0.0600]
  cells underpowered at TOLERANCE=0.05: 195 of 270
  original E9 operating point (predicted-positive rate): 0.1994


**Step 2. The figure.** Drawn only from the values printed above, then written to `fig-threshold-surface.pdf`.

In [ ]:
ROOT = pathlib.Path("/Users/rickygole/provenance-encoder")
d_thresh = json.loads((ROOT / "results/accord/selection/"
                       "threshold_robustness_results.json").read_text())
df = pd.DataFrame(d_thresh["cells"])
piv = df.pivot_table(index="achieved_rate", columns="beta",
                     values="demographic_parity_gap", aggfunc="mean")
mde = df.pivot_table(index="achieved_rate", columns="beta",
                     values="mde_demographic_parity_gap", aggfunc="mean")
R, B, Z, M = piv.index.values, piv.columns.values, piv.values, mde.values
e9_rate = d_thresh["original_e9_population_pred_pos_rate"]

# Named setting: the audit's own demographic-parity tolerance.
TOLERANCE = 0.05

plt.rcParams["hatch.color"] = SLATE
plt.rcParams["hatch.linewidth"] = 0.7

fig = plt.figure(figsize=(FIGW, 2.80))
canvas(fig)
ax = fig.add_axes([0.098, 0.212, 0.612, 0.578])
cax = fig.add_axes([0.898, 0.212, 0.028, 0.578])
im = ax.pcolormesh(B, R, Z, cmap=plt.get_cmap("viridis"), shading="nearest",
                   vmin=0, vmax=0.20)

under = (M > TOLERANCE).astype(float)
# A different channel, not a weaker version of the same one.
ax.contourf(B, R, under, levels=[0.5, 1.5], colors=[MIST], zorder=2)
ax.contourf(B, R, under, levels=[0.5, 1.5], colors=["none"], hatches=["////"],
            zorder=2.1)
ax.contour(B, R, under, levels=[0.5], colors=[GRAPHITE], linewidths=1.2,
           zorder=3)
# THE SMALL CAP ON THE BOTTOM EDGE NEAR beta = 0.5 IS DATA, NOT A LABELLING
# ARTIFACT. At the lowest predicted-positive rate the parity gap runs
# 0.0132, 0.0543, 0.0295 across beta = 0, 0.5, 1.0, so it crosses the 0.05
# tolerance up and back down inside one column. contour returns it as its own
# three-point segment, [(0.448, 0.090), (0.500, 0.106), (0.586, 0.090)], and it
# is drawn for the same reason the long branches are. Verified by rendering
# allsegs with no labelling of any kind in the figure. Do not delete it.
cs_tol = ax.contour(B, R, Z, levels=[TOLERANCE], colors=[GRAPHITE],
                    linewidths=1.2, zorder=4)

ax.set_xlabel(r"Selection strength through documentation volume ($\beta$)")
ax.set_ylabel("Predicted-positive rate")
ax.set_xticks([0, 2, 4, 6, 8])
ax.set_yticks([0.1, 0.2, 0.3, 0.4, 0.5, 0.6])
ax.set_xlim(B.min(), B.max())
ax.set_ylim(R.min(), R.max())
frame(ax)

# The E9 operating point: the one scarlet element, and it is the claim.
ax.plot(B, [e9_rate] * len(B), color=SCARLET_CORE, lw=LW_DATA, zorder=6)
case(ax.text(B[-1], e9_rate + 0.012, "the audit's operating point",
             ha="right", va="bottom", fontsize=T_SERIES, color=SCARLET_DEEP,
             zorder=7))


def exit_y(cset):
    """The y at which a contour leaves the plot on the right, so its label can
    sit in the margin beside it instead of riding the line."""
    best = None
    for seg in cset.allsegs[0]:
        if len(seg) == 0:
            continue
        i = int(np.argmax(seg[:, 0]))
        if best is None or seg[i, 0] > best[0]:
            best = (seg[i, 0], seg[i, 1])
    return None if best is None else best[1]


def exit_x_top(cset, y_top, tol=1e-9):
    """The x values at which a contour leaves the plot through the TOP edge.

    The tolerance contour never reaches the right-hand spine: both of its
    branches run off the top, at beta = 0.73 and beta = 2.59. Labelled at the
    right margin it pointed at a spine the line does not touch, so the label
    goes above the top edge at the branch's own exit x instead.
    """
    out = []
    for seg in cset.allsegs[0]:
        for x, y in seg:
            if abs(y - y_top) <= tol + 1e-12:
                out.append(float(x))
    return sorted(out)


AX_X0, AX_Y0, AX_W, AX_H = 0.098, 0.212, 0.612, 0.578
y_pow = exit_y(ax.contour(B, R, under, levels=[0.5], colors=["none"],
                          linewidths=0, zorder=0))
margin_x = 0.718
if y_pow is not None:
    yf = AX_Y0 + AX_H * (y_pow - R.min()) / (R.max() - R.min())
    fig.text(margin_x, yf, "power limit", ha="left", va="center",
             fontsize=T_SERIES, color=GRAPHITE)
    fig.patches.append(plt.Rectangle(
        (0.710, yf - 0.0022), 0.006, 0.0044, transform=fig.transFigure,
        facecolor=GRAPHITE, edgecolor="none", zorder=8))

tol_x = exit_x_top(cs_tol, R.max())
if tol_x:
    # The rightmost of the two top exits: the left one at beta = 0.73 would
    # centre a 51pt label off the left spine.
    xt = tol_x[-1]
    xf = AX_X0 + AX_W * (xt - B.min()) / (B.max() - B.min())
    fig.patches.append(plt.Rectangle(
        (xf - 0.0011, AX_Y0 + AX_H), 0.0022, 3.0 / (72 * 2.80),
        transform=fig.transFigure, facecolor=GRAPHITE, edgecolor="none",
        zorder=8))
    fig.text(xf, AX_Y0 + AX_H + 5.0 / (72 * 2.80), f"{TOLERANCE} tolerance",
             ha="center", va="bottom", fontsize=T_SERIES, color=GRAPHITE)

# The inability statement is carried on canvas by the headline and by the
# hatching itself; the sentence naming the hatched region is the caption's job
# and the caption says it verbatim (msu_ch5_discussion.tex). It was set twice.

cb = fig.colorbar(im, cax=cax)
cb.set_label("Demographic-parity gap", fontsize=T_AXIS)
cb.set_ticks([0, 0.05, 0.10, 0.15, 0.20])
cb.ax.tick_params(labelsize=T_TICK, width=LW_SPINE, length=3.2)
cb.outline.set_visible(True)
cb.outline.set_linewidth(LW_SPINE)
cb.outline.set_edgecolor(INK)

signature(fig, SCARLET_DEEP,
          "The audit passed at an operating point where nothing was detectable.")
out = HERE / "fig-threshold-surface.pdf"
fig.savefig(out)
n_under = int((M > TOLERANCE).sum())
print(f"wrote {out.name}  ({Z.shape[0]}x{Z.shape[1]} cells, {n_under} underpowered)")


## Figure 11: `fig-disc-actionability`


In [29]:
print("fig-disc-actionability: no result file is read by this cell.")
print("This figure is a schematic of a MIMIC-III field-availability fact (accord/CORRECTIONS.md, C6):")
print("  nursing notes carry CHARTTIME (a timestamp); discharge summaries carry only CHARTDATE (a date).")
print("No numeric estimate is plotted, so there is no CI to attach and nothing to source from a JSON file.")

# Widened from 10.5 to 11.6 units so the longest event label no longer runs off
# the right-hand edge, and every leader now starts outside the shaded band
# instead of crossing it. The trigger point carries one marker, not an X drawn
# on top of a dot.

fig-disc-actionability: no result file is read by this cell.
This figure is a schematic of a MIMIC-III field-availability fact (accord/CORRECTIONS.md, C6):
  nursing notes carry CHARTTIME (a timestamp); discharge summaries carry only CHARTDATE (a date).
No numeric estimate is plotted, so there is no CI to attach and nothing to source from a JSON file.


**Step 2. The figure.** Drawn only from the values printed above, then written to `fig-disc-actionability.pdf`.

In [ ]:
W, H = FIGW, 1.95
fig, ax = plt.subplots(figsize=(W, H))
canvas(fig)
plate(fig)
fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
ax.set_xlim(0, W * 72)
ax.set_ylim(0, H * 72)
ax.axis("off")

x_nurse, x_order, x_disch, x_summary = 48.0, 150.0, 268.0, 336.0
x0, x1 = 1.5, W * 72 - 1.5
g_lo, g_hi = 10.0, 112.0
band_lo, band_hi = 52.0, 68.0          # the 16pt band
mid = (band_lo + band_hi) / 2

phases = [
    (x0, x_order, EMERALD_WASH, EMERALD_TINT, EMERALD_DEEP,
     "intervention possible"),
    (x_order, x_disch, AMBER_WASH, AMBER_TINT, AMBER_DEEP,
     "window closing"),
    (x_disch, x1, SCARLET_WASH, SCARLET_TINT, SCARLET_DEEP,
     "no intervention possible"),
]
for a, b, wash, tint, deep, name in phases:
    ax.add_patch(mpatches.Rectangle((a, g_lo), b - a, g_hi - g_lo,
                                    facecolor=wash, edgecolor="none",
                                    linewidth=0, zorder=0))
for a, b, wash, tint, deep, name in phases:
    ax.add_patch(mpatches.Rectangle((a, band_lo), b - a, band_hi - band_lo,
                                    facecolor=tint, edgecolor=deep,
                                    linewidth=LW_NODE, zorder=2))
for a, b, wash, tint, deep, name in phases:
    case(ax.text((a + b) / 2, mid, name, ha="center", va="center",
                 fontsize=T_NOTE, color=INK, zorder=6))

# Event ticks pierce the band; names above it on two staggered levels, so
# no two names share a line and none needs a leader.
# The tick meets the band at both edges rather than running through it: the
# inside of the band belongs to the phase name, and a tick crossing that
# name is the one collision this layout can produce.
events = [
    (x_nurse, "last nursing note", GRAPHITE, 7.0, "center"),
    (x_order, "discharge order", GRAPHITE, 22.0, "center"),
    (x_disch, "the patient leaves", GRAPHITE, 7.0, "center"),
    (x_summary, "discharge summary finalised", SCARLET_DEEP, 22.0, "right"),
]
for x, label, tone, dy, ha in events:
    for lo, hi in ((band_lo - 6.0, band_lo), (band_hi, band_hi + 6.0)):
        ax.plot([x, x], [lo, hi], color=tone, lw=LW_CONNECTOR,
                solid_capstyle="butt", zorder=5)
    ax.text(x1 - 4.0 if ha == "right" else x, band_hi + dy, label, ha=ha,
            va="bottom", fontsize=T_SERIES, color=tone, zorder=6)

y_mark = band_lo - 9.0
y_note = band_lo - 20.0
# The point a detector could have fired: a proposal that was measured, so a
# CORE mark in the entity blue rather than an accent.
ax.plot([x_order], [y_mark], "o", color=COBALT_CORE, markersize=MS_MARK,
        markeredgecolor=COBALT_DEEP, markeredgewidth=1.0, zorder=7)
ax.text(x_order, y_note, "proposed detection point", ha="center", va="top",
        fontsize=T_SERIES, color=COBALT_DEEP, zorder=7)

# The system's trigger point: the one saturated scarlet mark on the canvas.
ax.plot([x_summary], [y_mark], "D", color=SCARLET_DEEP, markersize=9.4,
        markeredgecolor=SCARLET_CORE, markeredgewidth=LW_CHIP, zorder=7)
ax.text(x_summary, y_mark, "!", ha="center", va="center", fontsize=T_NOTE,
        color=PAPER, fontweight="bold", zorder=8)
ax.text(x1 - 4.0, y_note, "the system triggers here", ha="right", va="top",
        fontsize=T_SERIES, color=SCARLET_DEEP, zorder=7)

signature(fig, SCARLET_DEEP,
          "The document the system triggers on is finalised at or "
          "after the patient leaves.")
fname = "fig-disc-actionability.pdf"
fig.savefig(os.path.join(FIGDIR, fname))
plt.close(fig)
register(fname, "fig:disc-actionability", ["accord/CORRECTIONS.md (C6)"],
         "Structural/schema diagram, not derived from a numeric result file; documents a "
         "MIMIC-III field-availability fact. No CI applies (no estimate is plotted).")


## Figure 12: `fig-met-exchange`


In [31]:
p_baselines = os.path.join(RESULTS_ACCORD, "diag", "step3_baselines.json")
d_baselines = json.load(open(p_baselines))
systems = {
    "ACCORD": d_baselines["accord_binary_at_0.5_253core"],
    "GPT-4o zero-shot": d_baselines["gpt4o_zeroshot"],
    "GPT-4o few-shot": d_baselines["gpt4o_fewshot"],
    "Llama 3 8B": d_baselines["llama_baseline"],
}
ref = systems["ACCORD"]
TP0, TN0, FP0, FN0 = ref["tp"], ref["tn"], ref["fp"], ref["fn"]
P = TP0 + FN0
N = TN0 + FP0

def mcc_from_counts(tp, tn, fp, fn):
    num = tp * tn - fp * fn
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return 0.0 if den == 0 else num / den

mcc0 = mcc_from_counts(TP0, TN0, FP0, FN0)

print("fig-met-exchange: values plotted")
print(f"  reference system (ACCORD): TP={TP0}, TN={TN0}, FP={FP0}, FN={FN0}  (P={P}, N={N}, n={P + N})")
print(f"  reference MCC: {mcc0:.4f}")
for name, s in systems.items():
    delta = TP0 - s["tp"]
    dtn = s["tn"] - TN0
    print(f"  {name:18s}: TP={s['tp']:3d} TN={s['tn']:3d} FP={s['fp']:3d} FN={s['fn']:3d}  delta_TP={delta:+d}  delta_TN={dtn:+d}")

fig-met-exchange: values plotted
  reference system (ACCORD): TP=11, TN=18, FP=223, FN=1  (P=12, N=241, n=253)
  reference MCC: -0.0070
  ACCORD            : TP= 11 TN= 18 FP=223 FN=  1  delta_TP=+0  delta_TN=+0
  GPT-4o zero-shot  : TP=  6 TN=176 FP= 65 FN=  6  delta_TP=+5  delta_TN=+158
  GPT-4o few-shot   : TP=  7 TN=107 FP=134 FN=  5  delta_TP=+4  delta_TN=+89
  Llama 3 8B        : TP= 11 TN= 10 FP=231 FN=  1  delta_TP=+0  delta_TN=-8


**Step 2. The figure.** Drawn only from the values printed above, then written to `fig-met-exchange.pdf`.

In [ ]:
baselines, p_baselines = load("diag", "step3_baselines.json")
systems = {
    "ACCORD": baselines["accord_binary_at_0.5_253core"],
    "GPT-4o zero-shot": baselines["gpt4o_zeroshot"],
    "GPT-4o few-shot": baselines["gpt4o_fewshot"],
    "Llama 3 8B": baselines["llama_baseline"],
}
ref = systems["ACCORD"]
TP0, TN0, FP0, FN0 = ref["tp"], ref["tn"], ref["fp"], ref["fn"]
P = TP0 + FN0
N = TN0 + FP0
mcc0 = mcc_from_counts(TP0, TN0, FP0, FN0)

fig, ax = plt.subplots(figsize=(FIGW, 3.10))
canvas(fig)
fig.subplots_adjust(left=0.118, right=0.760, top=0.800, bottom=0.150)
# Zoomed to the occupied wedge. The old range ran to x = 12 only to give the
# two long leaders somewhere to go, and the leaders are gone. Starting at a
# negative x is what pulls the boxed left spine off the origin cluster.
X_LO, X_HI = -1.2, 7.0
# Y_HI is the FRAME TOP, not the data maximum. set_yticks() below unions the
# view interval with its own ticks, so a 200 tick pulls the top of the box to
# 200 whatever set_ylim said; building the region polygons to a smaller
# number left a white strip across the plot under the frame top. One
# constant, used by the limits, the ticks and the fills alike.
Y_LO, Y_HI = -30.0, 200.0
ax.set_xlim(X_LO, X_HI)
ax.set_ylim(Y_LO, Y_HI)

d = np.linspace(0, X_HI, 200)
slope_ba = N / P
# The indifference lines are not measurements: they are what each statistic
# would accept, computed from the class counts. What a reader needs is the
# SPAN of what different statistics will trade, so the span is filled.
deltas = np.linspace(0, min(TP0, X_HI), 60)
dtn_curve = []
for delta in deltas:
    tp = TP0 - delta
    fn = P - tp

    def f(tn):
        fp = N - tn
        return mcc_from_counts(tp, tn, fp, fn) - mcc0

    try:
        dtn_curve.append(brentq(f, 0, N) - TN0)
    except ValueError:
        dtn_curve.append(np.nan)
dtn_curve = np.array(dtn_curve)

# Three regions, and between them they are the whole content of this
# figure. Above the balanced-accuracy slope every statistic on this canvas
# counts the trade as a gain; below the PABAK slope none of them does;
# between the two it depends entirely on which statistic was chosen, and
# that span is the thing a reader has to see. Wider region first, narrower
# over it. HAIR under MIST rather than MIST under something darker: there
# is no structural stop between MIST and SLATE, and SLATE at area scale is
# a mark colour, not a ground.
dfull = np.linspace(X_LO, X_HI, 200)
ax.fill_between(dfull, np.clip(dfull * slope_ba, Y_LO, Y_HI), Y_HI,
                facecolor=HAIR, edgecolor="none", linewidth=0, zorder=1)
ax.fill_between(dfull, Y_LO, np.clip(dfull, Y_LO, Y_HI), facecolor=HAIR,
                edgecolor="none", linewidth=0, zorder=1)
ax.fill_between(dfull, np.clip(dfull, Y_LO, Y_HI),
                np.clip(dfull * slope_ba, Y_LO, Y_HI), facecolor=MIST,
                edgecolor="none", linewidth=0, zorder=2)
case(ax.text(6.7, 26.0, "only some of these statistics\n"
                        "accept a trade in this wedge",
             ha="right", va="center", fontsize=T_NOTE, color=MUTE,
             linespacing=1.24, zorder=4))
ax.plot(d, d * slope_ba, color=GRAPHITE, lw=LW_REF, zorder=3)
ax.plot(d, d, color=GRAPHITE, lw=LW_REF, zorder=3)
ax.plot(deltas, dtn_curve, color=GRAPHITE, lw=LW_REF, zorder=3)
# The x = 0 reference, now clearly separate from the boxed left spine.
ax.axvline(0, color=SLATE, linestyle=REF_DASH, lw=LW_REF, zorder=2)

# Each slope is labelled at the right edge of the plot at its own exit y,
# horizontal. Where two exit y values fall within 11pt the lower one drops
# to the next slot and takes a 3pt tick at the frame; a tick at the frame
# is not a leader.
exits = [
    (min(X_HI * slope_ba, Y_HI), f"balanced accuracy (slope {slope_ba:.1f})"),
    (X_HI, "PABAK (slope 1)"),
    (float(dtn_curve[-1]), f"MCC through ACCORD ({mcc0:.3f})"),
]
exits.sort(key=lambda e: -e[0])
pt_per_y = (0.800 - 0.150) * 3.10 * 72 / (Y_HI - Y_LO)
placed_y = []
for y, text in exits:
    yy = y
    for prev in placed_y:
        if abs(yy - prev) * pt_per_y < 11.0:
            yy = prev - 11.0 / pt_per_y
    placed_y.append(yy)
    ax.plot([X_HI - 0.10, X_HI], [yy, yy], color=GRAPHITE, lw=1.0,
            clip_on=False, zorder=5)
    ax.text(X_HI + 0.18, yy, text, ha="left", va="center",
            fontsize=T_SERIES, color=GRAPHITE, clip_on=False, zorder=5)

# The four systems. ACCORD is the claim: this whole figure is about where
# ACCORD sits against exchange rates it never chose, so it is the one solid
# scarlet mark and it gives up its amber on this canvas.
spec = {
    "ACCORD": ("D", SCARLET_DEEP, SCARLET_CORE, SCARLET_DEEP, 7.0,
               (-6, 7), "right", "bottom"),
    "Llama 3 8B": ("D", PAPER, PLUM_CORE, PLUM_DEEP, MS_MARK,
                   (8, -6), "left", "top"),
    "GPT-4o zero-shot": ("s", PAPER, COBALT_CORE, COBALT_DEEP, MS_MARK,
                         (7, 4), "left", "bottom"),
    "GPT-4o few-shot": ("^", PAPER, AZURE_CORE, AZURE_DEEP, MS_MARK,
                        (7, -5), "left", "top"),
}
for name, s_ in systems.items():
    delta = TP0 - s_["tp"]
    dtn = s_["tn"] - TN0
    mk, face, edge, tone, ms, off, ha, va = spec[name]
    ax.plot([delta], [dtn], mk, markerfacecolor=face, markeredgecolor=edge,
            markeredgewidth=LW_CHIP if name == "ACCORD" else 1.2, ms=ms,
            zorder=6)
    case(ax.annotate(name, (delta, dtn), textcoords="offset points",
                     xytext=off, ha=ha, va=va, fontsize=T_SERIES,
                     color=tone, zorder=7))

ax.set_xlabel(r"$\delta$ = forgone true positives, relative to ACCORD")
ax.set_ylabel(r"$\Delta$TN, relative to ACCORD")
ax.set_xticks([0, 2, 4, 6])
ax.set_yticks([0, 50, 100, 150, Y_HI])
frame(ax)
signature(fig, SCARLET_DEEP,
          "ACCORD sits on the indifference line: it buys no trade at all.")
fname = "fig-met-exchange.pdf"
fig.savefig(os.path.join(FIGDIR, fname))
plt.close(fig)
register(fname, "fig:met-exchange", [p_baselines],
         "Indifference lines are closed-form (PABAK, balanced accuracy) or numerically "
         "root-solved (MCC, scipy.optimize.brentq) from the confusion-matrix class counts "
         "in diag/step3_baselines.json; not fabricated curve shapes.")


## Final: verify all twelve exist


In [ ]:
EXPECTED = [
    "fig-pipeline-status.pdf",
    "fig-intro-headline.pdf",
    "fig-cohort-flow.pdf",
    "fig-exemplar.pdf",
    "fig-prevalence-response.pdf",
    "fig-falsification.pdf",
    "fig-power.pdf",
    "fig-forest.pdf",
    "fig-scaling.pdf",
    "fig-threshold-surface.pdf",
    "fig-disc-actionability.pdf",
    "fig-met-exchange.pdf",
]
missing = [f for f in EXPECTED if not os.path.exists(os.path.join(FIGDIR, f))]
print("figures written:", len(EXPECTED) - len(missing), "of", len(EXPECTED))
for f in missing:
    print("  MISSING", f)
assert not missing, missing
